# ARC-AGI-2 v13：NVARC 單輪 pass A＋Qwen3.8-27B 程式合成

1. NVARC（v11 原碼不改；以環境變數關閉 pass B）跑完 pass A，組出 submission.json。
2. 釋放 GPU 後，在獨立的 Python 3.12 venv 以 vLLM 0.19.0 服務 Qwen3.8-27B-FP8（TP4、MTP2），讓模型寫 `transform(grid)`；只有在增強後的全部訓練對上驗證通過的程式，其 test 答案才放進 attempt_1（attempt_2＝NVARC 第一名）。每個通過的程式都立即原子改寫 submission。
3. 任一步失敗都保留 NVARC 的 submission。Commit 跑＝開發量測（4 題 NVARC 冒煙＋keys30 程式工作）。

In [ ]:
import time
global_start = time.time()

In [ ]:
!pip uninstall -y tensorflow

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%writefile arc_v6_config.py
"""v6 runtime configuration: path discovery, rerun gating, task selection, deadlines.

No task IDs, hashes or counts of the public test file are pinned. Rerun mode takes every
task in the mounted test file; commit mode smoke-tests a few evaluation tasks chosen by
observed size only.
"""
import glob
import json
import os
import time

COMPETITION_SLUG = "arc-prize-2026-arc-agi-2"
MODEL_KEY = "qwen3_4b_grids15_sft139"
INPUT_ROOT = os.environ.get("ARC_V6_INPUT_ROOT", "/kaggle/input")
WORK_DIR = os.environ.get("ARC_V6_WORK_DIR", "/kaggle/working")
RERUN_WALL_SECONDS = 12 * 3600 - 600   # author global_end_time margin
COMMIT_WALL_SECONDS = 3600             # 4-task smoke; bounds the charged commit run
TASK_SECONDS = 1200                    # author per-task decode cap
TASK_FLOOR_SECONDS = 360               # v7: adaptive cap never below this (TTT + one decode batch)
COMMIT_TASK_SECONDS = 900              # v7: lower commit-smoke cap so the <TASK_SECONDS branch runs on hardware
SMOKE_TASKS = 4
NUM_WORKERS = 4
# v8: pass B re-runs tasks with independent seeds in the wall time pass A leaves unused; its candidates are
# pooled at assembly with pass A's top-1 always kept. Pass A itself is unchanged from v7.
PASS_B_MIN_SECONDS = float(os.environ.get("ARC_V8_PASS_B_MIN_SECONDS", "1200"))  # reload + floor + a task
# Pass B ends this much before end_time, so its kill + the two-store assembly stay well inside the 12 h limit.
PASS_B_DEADLINE_MARGIN = float(os.environ.get("ARC_V8_PASS_B_DEADLINE_MARGIN", "300"))
# v11: both passes run the luxluxshan half-cost width: 8 train augmentations x 8 geometries (64 TTT sequences)
# and 1 eval augmentation x 8 geometries (8 views), instead of the author's 16 / 2 (128 sequences, 16 views).
# Colab gate 2026-10-08 (colab-half-v1): half A took 0.555x full A; pooled half A+B pass@2 5.0 vs full A 3.0.
N_TRAIN_AUG = int(os.environ.get("ARC_V11_N_TRAIN_AUG", "8"))
N_EVAL_AUG = int(os.environ.get("ARC_V11_N_EVAL_AUG", "1"))
PASS_B_ENV = {"ARC_V8_TRAIN_AUG_SEED": "17", "ARC_V8_EVAL_AUG_SEED": "29", "ARC_V8_TRAINER_SEED": "137",
              "ARC_V8_LORA_SEED": "137", "ARC_V8_SCORE_SEED_OFFSET": "7"}
# v9: pass B first redraws the tasks pass A is unsure of that a floor cap would have cut, and only as many as
# their predicted full time fits (Colab pairing 2026-10-06: a second seed at cap 1200 solved a mid-size task pass A
# missed; at the 360 s floor it never did; tasks under ~360 s already got full redraws in v8).
PASS_B_CONFIDENT = float(os.environ.get("ARC_V9_PASS_B_CONFIDENT", "5"))  # kgmon top-1 score counted as sure
PASS_B_PLAN_SECONDS = float(os.environ.get("ARC_V9_PASS_B_PLAN_SECONDS", "120"))  # bound on reading store A
PASS_B_PLAN_FACTOR = 0.555  # v11: half-cost / full-cost seconds (Colab gate); v6-v8 Kaggle smokes ran at 0.5-1.0x the fit
PASS_B_START_STAGGER = 180  # pass-B ranks import one at a time (~44 s each) before taking tasks
PASS_B_PLAN_SCALE = float(os.environ.get("ARC_V9_PASS_B_PLAN_SCALE", "1"))  # test seam only; production = 1
SECONDS_FIT = (-223.08392554483123, 0.3000408772151657, -1.550904629974504e-05)  # colab-cal-v1 one-L4, cap 1200


def is_rerun():
    # Same truthiness as the author notebook.
    return bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))


def _single(paths, prefer, what):
    paths = sorted(set(paths))
    preferred = [p for p in paths if prefer in p.lower()]
    paths = preferred or paths
    if len(paths) != 1:
        raise RuntimeError(f"expected exactly one {what}, found {len(paths)}: {paths[:5]}")
    return paths[0]


def competition_dir(root=None):
    root = root or INPUT_ROOT
    hits = glob.glob(os.path.join(root, "**", "arc-agi_test_challenges.json"), recursive=True)
    return os.path.dirname(_single(hits, COMPETITION_SLUG, "competition test file"))


def model_dir(root=None):
    root = root or INPUT_ROOT
    hits = [os.path.dirname(p) for p in glob.glob(os.path.join(root, "**", "config.json"), recursive=True)
            if MODEL_KEY in p.lower()]
    return _single(hits, MODEL_KEY, "model directory")


def test_path(root=None):
    return os.path.join(competition_dir(root), "arc-agi_test_challenges.json")


def work_challenges_path(root=None):
    """Challenge file the model workers process in this mode."""
    name = "arc-agi_test_challenges.json" if is_rerun() else "arc-agi_evaluation_challenges.json"
    return os.path.join(competition_dir(root), name)


def _cells(task):
    return sum(len(g) * len(g[0]) for pairs in task.values() for pair in pairs for g in pair.values() if g)


def select_keys(challenges, rerun=None):
    """Rerun: every task, cheapest estimated work first (v7). Commit: size-chosen smoke set."""
    rerun = is_rerun() if rerun is None else rerun
    keys = sorted(challenges)
    if rerun:
        return sorted(keys, key=lambda k: (estimated_work(challenges[k]), k))
    by_size = sorted(keys, key=lambda k: (_cells(challenges[k]), k))
    chosen = [by_size[-1]]                                   # largest context: VRAM/length check
    multi = [k for k in by_size if len(challenges[k]["test"]) > 1 and k not in chosen]
    chosen += multi[:1]                                      # multiple test inputs
    chosen += [k for k in by_size if k not in chosen][:SMOKE_TASKS - len(chosen)]
    return sorted(chosen)


def estimated_work(task):
    """Size proxy: TTT cost grows with all grid cells, decoding with the number of test inputs."""
    try:
        return _cells(task) * (1 + 0.5 * (len(task["test"]) - 1))
    except Exception:  # an unexpected task shape must never stop the queue from being built
        return 0


def adaptive_cap(remaining_seconds, pending_tasks):
    """Fair share of the remaining wall over pending + in-flight tasks, within [floor, author cap]."""
    share = remaining_seconds * NUM_WORKERS / (max(0, pending_tasks) + NUM_WORKERS)
    upper = TASK_SECONDS if is_rerun() else COMMIT_TASK_SECONDS
    return max(TASK_FLOOR_SECONDS, min(upper, share))


def seed(name, default):
    """v8: per-pass seed; unset (pass A) keeps the v7 value."""
    return int(os.environ.get(name, default))


def outputs_dir():
    return os.environ.get("ARC_V6_OUTPUTS", "/kaggle/inference_outputs")


def outputs_dir_b():
    return os.environ.get("ARC_V8_OUTPUTS_B", outputs_dir().rstrip("/") + "_b")


def output_task(name):
    """Task key of a decoded-result file name ('<task>_<i>.<augmentation>')."""
    return name.split(".")[0].rsplit("_", 1)[0]


def covered_tasks(outputs):
    """Tasks with at least one finished decoded-result file in a store."""
    try:
        names = os.listdir(outputs)
    except OSError:
        names = []
    return {output_task(n) for n in names if not n.startswith("._tmp_")}


def failed_path(outputs):
    return outputs.rstrip("/") + "_failed.txt"


def mark_failed(key):
    """v8: record a task whose solve raised, so pass B tries it last; never raises."""
    try:
        with open(failed_path(outputs_dir()), "a") as f:
            f.write(json.dumps(key) + "\n")
    except Exception:
        pass


def failed_tasks(outputs):
    try:
        with open(failed_path(outputs)) as f:
            return {json.loads(line) for line in f if line.strip()}
    except Exception:
        return set()


def pass_b_keys(keys, outputs_a):
    """Pass B order (pass-A order kept within groups): tasks pass A left without any decoded result and that
    did not raise, then tasks with results, then tasks that raised without results (likely to fail again)."""
    covered, failed = covered_tasks(outputs_a), failed_tasks(outputs_a)
    missing = [k for k in keys if k not in covered and k not in failed]
    return missing + [k for k in keys if k in covered] + [k for k in keys if k not in covered and k in failed]


def task_confidence(keys, outputs_a, challenges, deadline):
    """v9: per task, the minimum over its test outputs of pass A's kgmon top-1 score; an output without any
    readable candidate counts as -inf (unsure). Raises TimeoutError past the deadline."""
    import bz2
    import pickle
    from arc_decoder import getter_kgmon, hashable
    groups = {}
    for name in sorted(os.listdir(outputs_a)):
        if time.time() > deadline:
            raise TimeoutError("reading pass-A store took too long")
        if name.startswith("._tmp_"):
            continue
        try:
            with bz2.BZ2File(os.path.join(outputs_a, name)) as f:
                samples = pickle.load(f)
        except Exception:
            continue  # same tolerance as the assembly loader
        by_grid = groups.setdefault(name.split(".")[0], {})
        for sample in samples:
            try:
                by_grid.setdefault(hashable(sample["solution"]), []).append(sample)
            except Exception:
                pass
    conf = {}
    for k in keys:
        scores = []
        for i in range(len(challenges[k]["test"])):
            best = float("-inf")
            for guesses in groups.get(f"{k}_{i}", {}).values():
                try:
                    best = max(best, float(getter_kgmon(guesses)))
                except Exception:
                    pass
            scores.append(best)
        conf[k] = min(scores) if scores else float("-inf")
    return conf


def task_tokens(task):
    """Prompt length the model sees (colab-cal-v1 analyze.tokens): train pairs + the largest test input + 900."""
    gl = lambda g: len(g) * (len(g[0]) + 1)
    return min(8192, sum(gl(p["input"]) + gl(p["output"]) + 10 for p in task["train"])
               + max(gl(q["input"]) for q in task["test"]) + 900)


def full_seconds(task):
    """v9: predicted Kaggle seconds for one uncut solve (TTT + decode); an odd shape plans the author cap."""
    try:
        L = task_tokens(task)
        return (SECONDS_FIT[0] + SECONDS_FIT[1] * L + SECONDS_FIT[2] * L * L) * PASS_B_PLAN_FACTOR
    except Exception:
        return float(TASK_SECONDS)


def pass_b_plan(keys, outputs_a, challenges, seconds_left):
    """v9 pass-B queue, pass-A cheap-first order kept within every group: tasks pass A left without results; tasks
    pass A is unsure of whose full solve a floor cap would cut; the other unsure tasks (v8 already redrew those
    fully); sure tasks; tasks that raised. Truncated where the planned seconds (full solve, clipped to the
    [floor, author] caps) exceed the pass-B worker budget. Any error falls back to v8's full queue (pass_b_keys).
    Returns (keys, info)."""
    start = time.time()
    try:
        conf = task_confidence(keys, outputs_a, challenges, start + PASS_B_PLAN_SECONDS)
        covered, failed = covered_tasks(outputs_a), failed_tasks(outputs_a)
        full = {k: full_seconds(challenges[k]) for k in keys}
        missing = [k for k in keys if k not in covered and k not in failed]
        unsure = [k for k in keys if k in covered and not conf[k] >= PASS_B_CONFIDENT]
        long_ = [k for k in unsure if full[k] > TASK_FLOOR_SECONDS]
        short = [k for k in unsure if not full[k] > TASK_FLOOR_SECONDS]
        sure = [k for k in keys if k in covered and conf[k] >= PASS_B_CONFIDENT]
        raised = [k for k in keys if k not in covered and k in failed]
        budget = (seconds_left - (time.time() - start) - PASS_B_START_STAGGER * PASS_B_PLAN_SCALE) * NUM_WORKERS
        ordered, planned, total = missing + long_ + short + sure + raised, [], 0.0
        for k in ordered:
            need = min(TASK_SECONDS, max(TASK_FLOOR_SECONDS, full[k])) * PASS_B_PLAN_SCALE
            if len(planned) >= NUM_WORKERS and total + need > budget:
                break
            planned.append(k)
            total += need
        info = {"pass_b_order": "confidence", "missing": len(missing), "unsure_long": len(long_),
                "unsure_short": len(short), "sure": len(sure), "raised": len(raised), "limit": len(planned),
                "planned_sum": round(total), "budget": round(budget), "threshold": PASS_B_CONFIDENT,
                "plan_seconds": round(time.time() - start, 1)}
        return planned, info
    except Exception as exc:
        return pass_b_keys(keys, outputs_a), {"pass_b_order": "fallback", "error": f"{type(exc).__name__}: {exc}"[:300],
                                              "plan_seconds": round(time.time() - start, 1)}


def decode_batches(test_id_to_subkeys, length_of, batch_size=4):
    """Decode batches per test input. With the author's 16 views (8 geometries x 2 permutations, sorted subkeys)
    the author's fixed pairing is kept verbatim; any other view count (v11 half-cost: 8 views) batches prompts of
    equal token length (batched DFS needs equal lengths), at most batch_size each, in sorted subkey order."""
    if all(len(subkeys) == 16 for subkeys in test_id_to_subkeys.values()):
        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            for offsets in ([0, 4], [2, 6]):
                batches.append([sk for offset in offsets for sk in subkeys[offset:offset + 2]])
        for test_id, subkeys in test_id_to_subkeys.items():
            for offsets in ([8, 12], [10, 14]):
                batches.append([sk for offset in offsets for sk in subkeys[offset:offset + 2]])
        return batches
    batches = []
    for test_id, subkeys in test_id_to_subkeys.items():
        groups = {}
        for subkey in subkeys:
            groups.setdefault(length_of(subkey), []).append(subkey)
        for group in groups.values():
            batches.extend(group[i:i + batch_size] for i in range(0, len(group), batch_size))
    return batches


def end_time(start):
    return start + (RERUN_WALL_SECONDS if is_rerun() else COMMIT_WALL_SECONDS)


def atomic_json(path, value):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(value, f)
    os.replace(tmp, path)


def fallback_submission(test_challenges):
    return {k: [{"attempt_1": [[0]], "attempt_2": [[0]]} for _ in task["test"]]
            for k, task in test_challenges.items()}


def validate_submission(test_challenges, submission):
    if set(submission) != set(test_challenges):
        raise ValueError("submission keys differ from test keys")
    for k, task in test_challenges.items():
        entries = submission[k]
        if not isinstance(entries, list) or len(entries) != len(task["test"]):
            raise ValueError(f"{k}: wrong number of test entries")
        for e in entries:
            if not isinstance(e, dict) or set(e) != {"attempt_1", "attempt_2"}:
                raise ValueError(f"{k}: attempts missing")
            if not all(valid_grid(g) for g in e.values()):
                raise ValueError(f"{k}: invalid grid")
    return True


def valid_grid(g):
    return (isinstance(g, list) and 1 <= len(g) <= 30 and isinstance(g[0], list) and 1 <= len(g[0]) <= 30
            and all(isinstance(r, list) and len(r) == len(g[0]) for r in g)
            and all(type(c) is int and 0 <= c <= 9 for r in g for c in r))


def log(message, **fields):
    print(json.dumps({"t": round(time.time(), 1), "msg": message, **fields}), flush=True)


In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
try:
    from transformers import AutoTokenizer
except ImportError:  # v6: CPU hidden-shape tests run without transformers
    AutoTokenizer = None


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.rsplit('_', 1)
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score


In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in sorted(os.listdir(store)):
            if key.startswith("._tmp_"):
                continue  # v6: unfinished atomic write
            try:
                with bz2.BZ2File(os.path.join(store, key)) as f:
                    outputs = pickle.load(f)
            except Exception as exc:  # v6: a truncated file must not drop the submission
                print(f"*** skip unreadable decoded result {key}: {type(exc).__name__}")
                continue
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")


In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter
import arc_v6_config

import gc
import traceback
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    batch_logits = outputs.logits.float().cpu().log_softmax(-1)
    result = []
    for logits, query_tokens, answer_tokens in zip(batch_logits, batch_query_tokens, batch_answer_tokens):
        query_length = len(query_tokens)
        answer_logits = logits[query_length-1:query_length-1+len(answer_tokens)]
        answer_score = answer_logits[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=arc_v6_config.seed("ARC_V8_LORA_SEED", 42),  # v8: pass B reseeds
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=arc_v6_config.seed("ARC_V8_TRAINER_SEED", 42),
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=arc_v6_config.model_dir(),
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    test_path = arc_v6_config.work_challenges_path()

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = arc_v6_config.outputs_dir()  # v8: pass B writes to its own store
    os.makedirs(dir_outputs, exist_ok=True)

    consecutive_failures = 0

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break

        # v7: a task picked up in the last TASK_FLOOR_SECONDS cannot finish TTT + one batch.
        if end_time - time.time() < arc_v6_config.TASK_FLOOR_SECONDS:
            print(f"[Rank {rank}] stop: {end_time - time.time():.0f}s left, below task floor", flush=True)
            break

        # v7: shrink the per-task cap when the remaining wall cannot cover the queue at 1200 s/task.
        # qsize still holds the NUM_WORKERS None sentinels while real tasks remain.
        task_cap = arc_v6_config.adaptive_cap(end_time - time.time(), queue.qsize() - arc_v6_config.NUM_WORKERS)
        print(f"[Rank {rank}] task {key} cap {task_cap:.0f}s", flush=True)

        failed = False
        try:
            model = solve_task(key, model, tokenizer, formatter, collator, default_weights, train_args,
                               arc_test_set, max_seq_length, max_new_tokens, max_score, dir_outputs, rank, end_time,
                               task_cap)
            consecutive_failures = 0
        except Exception as exc:  # v6: one failing task must not stop this GPU's queue
            failed = True
            # v7: an OOM on one large task is usually recoverable, so it weighs half (6 in a row retire the worker).
            consecutive_failures += 0.5 if isinstance(exc, torch.cuda.OutOfMemoryError) else 1
            print(f"[Rank {rank}] task {key} failed: {type(exc).__name__}: {str(exc)[:300]}", flush=True)
            arc_v6_config.mark_failed(key)  # v8: pass B tries tasks that raised last
            traceback.print_exc()
            # A sticky CUDA fault would make this worker drain the shared queue; leave it to the others.
            if "CUDA error" in str(exc) or consecutive_failures >= 3:
                print(f"[Rank {rank}] stopping after {consecutive_failures} failure(s)", flush=True)
                break
        if failed:  # v7: clean up after the except block releases the traceback (trainer/optimizer refs)
            gc.collect()
            torch.cuda.empty_cache()


def solve_task(key, model, tokenizer, formatter, collator, default_weights, train_args,
               arc_test_set, max_seq_length, max_new_tokens, max_score, dir_outputs, rank, end_time, task_cap=arc_v6_config.TASK_SECONDS):
        start_time = time.time()
        decoded_test_ids = set()
        score_seed_offset = arc_v6_config.seed("ARC_V8_SCORE_SEED_OFFSET", 0)
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=arc_v6_config.N_TRAIN_AUG, shfl_keys=True, seed=arc_v6_config.seed("ARC_V8_TRAIN_AUG_SEED", 1))
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=arc_v6_config.N_EVAL_AUG, seed=arc_v6_config.seed("ARC_V8_EVAL_AUG_SEED", 2))
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].rsplit("_", 1)[1]
            test_id_to_subkeys[test_id].append(subkey)

        # v11: batching moved to arc_v6_config.decode_batches (author pairing kept verbatim for 16 views;
        # other view counts, e.g. half-cost 8 views, batch equal-length prompts).
        batches = arc_v6_config.decode_batches(
            test_id_to_subkeys, lambda sk: len(tokenizer.encode(eval_ds.get(sk, formatter)["input"])))

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                batch_test_id = subkeys[0].split(".")[0].rsplit("_", 1)[1]
                # v7: every test input gets at least one decode batch after (unbounded) TTT;
                # further batches respect the (adaptive) per-task cap.
                first_for_test = batch_test_id not in decoded_test_ids
                if time.time() > end_time or (spend_time > task_cap and not first_for_test):
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    if time.time() > end_time:
                        break
                    continue
                decoded_test_ids.add(batch_test_id)
                # Author behaviour at the full cap; below it, DFS stops at the task deadline but
                # always keeps a short window for a guaranteed first batch.
                dfs_end_time = end_time if task_cap >= arc_v6_config.TASK_SECONDS else min(
                    end_time, max(start_time + task_cap, time.time() + 180))

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, dfs_end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=(hash(bk) + score_seed_offset) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        tmp_path = os.path.join(dir_outputs, "._tmp_" + subkey)
                        with bz2.BZ2File(tmp_path, "w") as f:
                            pickle.dump(decoded_result, f)
                        os.replace(tmp_path, os.path.join(dir_outputs, subkey))  # v6: atomic

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")
        return model


In [ ]:
%%writefile starter.py
import argparse
import importlib
import json
import multiprocessing as mp
import os
import time

import arc_v6_config as cfg

MARKER_DIR = os.environ.get("ARC_V6_MARKER_DIR", "/kaggle")
SOLVER = os.environ.get("ARC_V6_SOLVER_MODULE", "arc_solver")  # test seam; production = arc_solver
MARKER_WAIT_SECONDS = 900
MARKER_POLL_SECONDS = float(os.environ.get("ARC_V6_MARKER_POLL", "5"))
KILL_GRACE_SECONDS = float(os.environ.get("ARC_V6_KILL_GRACE", "120"))


def marker(rank, label="A"):
    return os.path.join(MARKER_DIR, f"worker{rank}" + ("" if label == "A" else "." + label))


def local_worker(rank, queue, end_time, label="A", env=None):

    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    os.environ.update(env or {})  # v8: pass B seeds and output store, set before the solver is imported

    if SOLVER == "arc_solver":
        import torch
        torch.set_default_device("cpu")

    # Fix Unsloth patching issue (author): import ranks one at a time, but never wait forever.
    if rank > 0:
        waited = 0
        while not os.path.exists(marker(rank - 1, label)) and waited < MARKER_WAIT_SECONDS:
            time.sleep(MARKER_POLL_SECONDS)
            waited += MARKER_POLL_SECONDS

    worker = importlib.import_module(SOLVER).worker

    with open(marker(rank, label), "w") as f:
        f.write("Ok")

    print(f"[Rank {rank}] start!" + ("" if label == "A" else f" pass {label}"), flush=True)

    worker(rank, queue, end_time)

    print(f"[Rank {rank}] done!", flush=True)


def run_pass(ctx, manager, keys, end_time, label="A", env=None):
    """One pass: NUM_WORKERS independent GPU processes over a shared queue. True when every worker is gone."""
    queue = manager.Queue()
    for key in keys:
        queue.put(key)
    for _ in range(cfg.NUM_WORKERS):
        queue.put(None)

    # Independent processes: one crashed GPU worker must not kill the other three
    # (torch.multiprocessing.spawn terminates all ranks when one fails).
    procs = [ctx.Process(target=local_worker, args=(rank, queue, end_time, label, env)) for rank in range(cfg.NUM_WORKERS)]
    for p in procs:
        p.start()
    while any(p.is_alive() for p in procs) and time.time() < end_time + KILL_GRACE_SECONDS:
        time.sleep(2)
    for p in procs:
        if p.is_alive():
            p.terminate()
    for p in procs:
        p.join(15)
        if p.is_alive():
            p.kill()
            p.join(5)
    cfg.log("workers_exited", exitcodes=[p.exitcode for p in procs], pass_label=label)
    return not any(p.is_alive() for p in procs)


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    with open(cfg.work_challenges_path(), "r") as f:
        data = json.load(f)
    keys = cfg.select_keys(data)
    cfg.log("queue", rerun=cfg.is_rerun(), tasks=len(keys), challenges=len(data),
            seconds_left=round(args.end_time - time.time()))

    # v7: fixed hash seed so hash(bk)-seeded augmentation scoring is reproducible across runs.
    os.environ.setdefault("PYTHONHASHSEED", "0")
    ctx = mp.get_context("spawn")
    manager = ctx.Manager()

    clean = run_pass(ctx, manager, keys, args.end_time)

    # v8: pass B only in wall time pass A did not need; a saturated run behaves exactly like v7.
    end_b = args.end_time - cfg.PASS_B_DEADLINE_MARGIN
    seconds_left = end_b - time.time()
    if clean and seconds_left >= cfg.PASS_B_MIN_SECONDS:
        keys_b, plan = cfg.pass_b_plan(keys, cfg.outputs_dir(), data, seconds_left)  # v9: unsure tasks, near-full caps
        env_b = dict(cfg.PASS_B_ENV, ARC_V6_OUTPUTS=cfg.outputs_dir_b())
        missing = len(set(keys) - cfg.covered_tasks(cfg.outputs_dir()))
        cfg.log("pass_b_start", tasks=len(keys_b), missing_after_a=missing,
                failed_after_a=len(cfg.failed_tasks(cfg.outputs_dir())), seconds_left=round(end_b - time.time()), **plan)
        clean = run_pass(ctx, manager, keys_b, end_b, label="B", env=env_b)
    else:
        cfg.log("pass_b_skipped", clean=clean, seconds_left=round(seconds_left))
    if not clean:
        # An unkillable child would block multiprocessing's atexit join forever.
        cfg.log("abandoning_unkillable_workers")
        os._exit(0)


In [ ]:
%%writefile assemble.py
"""Final cell: merge decoded results into a full-coverage submission.json."""
import json
import os
import traceback

import numpy as np

import arc_v6_config as cfg
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

OUTPUTS = os.environ.get("ARC_V6_OUTPUTS", "/kaggle/inference_outputs")


def load_decoder(dataset, stores):
    decoder = ArcDecoder(dataset, n_guesses=2)
    for store, run_name in stores:
        if os.path.isdir(store):
            decoder.load_decoded_results(store, run_name=run_name)
    known = set(dataset.keys)
    stale = [k for k in decoder.decoded_results if k not in known]
    for k in stale:  # never let an unknown key break the merge
        del decoder.decoded_results[k]
    return decoder


def keep_a_top1(sel_a, sel_p):
    """v8: pass A's top-1 is always one of the two attempts; the other is the best distinct pooled grid.

    With no pass-B candidates for an output, the pooled ranking equals pass A's, so this is v7's choice.
    """
    selected, changed = {}, 0
    for bk in set(sel_a) | set(sel_p):
        ranked_a, ranked_p = list(sel_a.get(bk, [])), list(sel_p.get(bk, []))
        if not ranked_a:
            selected[bk] = ranked_p[:2]
        else:
            first = ranked_a[0]
            other = next((g for g in ranked_p + ranked_a[1:] if not np.array_equal(g, first)), None)
            selected[bk] = [first] + ([other] if other is not None else [])
        before = ranked_a[:2]
        changed += len(before) != len(selected[bk]) or any(
            not np.array_equal(x, y) for x, y in zip(before, selected[bk]))
    return selected, changed


def model_submission(challenges_path, outputs_dir, solutions_path=None, use_b=True):
    data = ArcDataset.from_file(challenges_path)
    if solutions_path:
        data = data.load_replies(solutions_path)
    multi = data.split_multi_replies()
    outputs_b = cfg.outputs_dir_b()
    decoder_a = load_decoder(multi, [(outputs_dir, "")])
    pooled = load_decoder(multi, [(outputs_dir, ""), (outputs_b, ".b")]) if use_b else decoder_a
    selected, changed = keep_a_top1(decoder_a.run_selection_algo(), pooled.run_selection_algo())
    stats = {"outputs_a": len(decoder_a.decoded_results), "outputs_pooled": len(pooled.decoded_results),
             "pass_b_files": len(os.listdir(outputs_b)) if use_b and os.path.isdir(outputs_b) else 0,
             "changed_vs_pass_a": int(changed)}
    return data, decoder_a, pooled, data.get_submission(selected), stats


def merge(test, fallback, candidate):
    """Per attempt: take the model grid if valid, else keep the fallback grid."""
    merged, used = {}, 0
    for key, task in test.items():
        entries = []
        for i in range(len(task["test"])):
            entry = dict(fallback[key][i])
            for attempt in ("attempt_1", "attempt_2"):
                try:
                    grid = np.asarray(candidate[key][i][attempt]).tolist()
                except Exception:
                    continue
                if cfg.valid_grid(grid):
                    entry[attempt] = grid
                    used += grid != [[0]]
            entries.append(entry)
        merged[key] = entries
    return merged, used


def main(work_dir=None, outputs_dir=OUTPUTS):
    work_dir = work_dir or cfg.WORK_DIR
    test_path = cfg.test_path()
    with open(test_path) as f:
        test = json.load(f)
    fallback = cfg.fallback_submission(test)
    final, used, error, pool_stats = fallback, 0, None, {}
    if cfg.is_rerun():
        try:
            try:
                _, _, _, candidate, pool_stats = model_submission(test_path, outputs_dir)
            except Exception:  # v8: a pass-B problem must never cost pass A's answers
                pool_stats = {"pooling_error": traceback.format_exc()[-1500:]}
                cfg.log("pooling_failed_using_pass_a")
                _, _, _, candidate, stats_a = model_submission(test_path, outputs_dir, use_b=False)
                pool_stats.update(stats_a)
            final, used = merge(test, fallback, candidate)
            cfg.validate_submission(test, final)
        except Exception:
            error = traceback.format_exc()
            final, used = fallback, 0
    else:
        # Commit smoke: model ran on evaluation tasks; report them separately, keep the
        # test-file submission full-coverage fallback so the version stays submittable.
        comp = cfg.competition_dir()
        sol = os.path.join(comp, "arc-agi_evaluation_solutions.json")
        try:
            data, decoder, pooled, candidate, pool_stats = model_submission(
                cfg.work_challenges_path(), outputs_dir, sol if os.path.exists(sol) else None)
            cfg.atomic_json(os.path.join(work_dir, "eval-smoke-submission.json"), candidate)
            if pooled.decoded_results and os.path.exists(sol):
                for label, dec in (("pass_a", decoder), ("pooled", pooled)):
                    try:
                        print(f"*** {label}")
                        dec.benchmark_selection_algos()
                    except Exception as exc:  # author benchmark fails when nothing is solved
                        cfg.log("benchmark_skipped", label=label, error=type(exc).__name__)
                ran = sorted({b.rsplit("_", 1)[0] for b in pooled.decoded_results})
                sel_a = decoder.run_selection_algo()
                solved_a = sum(np.mean([any(np.array_equal(r, g) for g in sel_a.get(f"{k}_{i}", [])[:2])
                                        for i, r in enumerate(data.replies[k])]) for k in ran)
                solved = sum(np.mean([any(np.array_equal(r, e[a]) for a in ("attempt_1", "attempt_2"))
                                      for r, e in zip(data.replies[k], candidate[k])]) for k in ran)
                cfg.log("eval_smoke_score", solved=float(solved), tasks=len(ran), solved_pass_a=float(solved_a),
                        contaminated_by_pretraining=True)
        except Exception:
            error = traceback.format_exc()
    cfg.validate_submission(test, final)
    cfg.atomic_json(os.path.join(work_dir, "submission.json"), final)
    record = {"rerun": cfg.is_rerun(), "test_tasks": len(test),
              "test_inputs": sum(len(t["test"]) for t in test.values()),
              "model_attempts_used": used, "error": error,
              "decoded_files": len(os.listdir(outputs_dir)) if os.path.isdir(outputs_dir) else 0, **pool_stats}
    cfg.atomic_json(os.path.join(work_dir, "v6-assembly.json"), record)
    cfg.log("assembled", **{k: v for k, v in record.items() if k != "error"}, error=bool(error))
    if error:
        print(error[-3000:])
    return record


if __name__ == "__main__":
    main()


In [ ]:
%%writefile arc_llm.py
"""Pure helpers for the Qwen3.8 ARC probe (new-approach-proposal-v1 §6): prompt, answer parsing, dihedral + colour
augmentation and its inverse, and summary statistics. No I/O, no third-party imports, so it runs in the probe and in
local unit tests alike."""
import json
import math
import random
import re
import zlib

# --- grids -----------------------------------------------------------------------------------------------------------

def valid_grid(g):
    return (isinstance(g, list) and 0 < len(g) <= 30 and all(isinstance(r, list) for r in g)
            and 0 < len(g[0]) <= 30 and all(len(r) == len(g[0]) for r in g)
            and all(type(v) is int and 0 <= v <= 9 for r in g for v in r))


def transpose(g):
    return [list(r) for r in zip(*g)]


def rot90(g):  # clockwise
    return [list(r) for r in zip(*g[::-1])]


def dihedral(g, k):
    """k in 0..7: rotate k%4 quarter turns clockwise, then transpose if k >= 4."""
    for _ in range(k % 4):
        g = rot90(g)
    return transpose(g) if k >= 4 else [list(r) for r in g]


def dihedral_inverse(g, k):
    if k >= 4:
        g = transpose(g)
    for _ in range((4 - k % 4) % 4):
        g = rot90(g)
    return [list(r) for r in g]


def recolor(g, perm):
    return [[perm[v] for v in r] for r in g]


def augmentation(task_id, salt="p0-v1"):
    """Deterministic non-identity transform per task: dihedral k in 1..7 and a permutation of colours 1..9 (0 fixed,
    ARC background)."""
    rng = random.Random(zlib.crc32(f"{salt}:{task_id}".encode()))
    k = rng.randrange(1, 8)
    rest = list(range(1, 10))
    rng.shuffle(rest)
    perm = [0] + rest
    return {"k": k, "perm": perm}


def apply_aug(grid, aug):
    return recolor(dihedral(grid, aug["k"]), aug["perm"])


def invert_aug(grid, aug):
    inv = [0] * 10
    for a, b in enumerate(aug["perm"]):
        inv[b] = a
    return dihedral_inverse(recolor(grid, inv), aug["k"])


def augment_task(task, aug):
    return {"train": [{"input": apply_aug(p["input"], aug), "output": apply_aug(p["output"], aug)}
                      for p in task["train"]],
            "test": [{"input": apply_aug(p["input"], aug)} for p in task["test"]]}


# --- prompt / parse --------------------------------------------------------------------------------------------------

def grid_text(g):
    return "[" + ",\n ".join(json.dumps(r, separators=(",", ":")) for r in g) + "]"


def build_prompt(task, test_index):
    parts = ["You are solving an ARC-AGI puzzle. Each example maps an input grid to an output grid with the same "
             "hidden rule. Grids are lists of rows; each cell is a colour 0-9.", ""]
    for i, p in enumerate(task["train"], 1):
        parts += [f"Example {i} input:", grid_text(p["input"]), f"Example {i} output:", grid_text(p["output"]), ""]
    parts += ["Test input:", grid_text(task["test"][test_index]["input"]), "",
              "Find the rule, apply it to the test input, and give the test output grid. End your answer with the "
              "output grid as a JSON list of lists inside a ```json code block, and nothing after it."]
    return "\n".join(parts)


_FENCE = re.compile(r"```(?:json)?\s*(\[.*?\])\s*```", re.S)


def _candidates(text):
    for m in _FENCE.finditer(text):
        yield m.group(1)
    # bare outermost [[...]] blocks, last ones last
    depth, start = 0, None
    for i, ch in enumerate(text):
        if ch == "[":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "]" and depth:
            depth -= 1
            if depth == 0 and start is not None:
                yield text[start:i + 1]


def parse_answer(text):
    """Last valid grid found, scanning fenced blocks first and then every bare outermost list, so a valid bare list after
    the fence wins; None if there is none."""
    if not text:
        return None
    found = None
    for chunk in _candidates(text):
        try:
            g = json.loads(chunk)
        except (ValueError, RecursionError):
            continue
        if valid_grid(g):
            found = g
    return found


# --- statistics ------------------------------------------------------------------------------------------------------

def quantiles(xs, qs=(0.5, 0.9)):
    if not xs:
        return {f"p{int(q * 100)}": None for q in qs}
    s = sorted(xs)
    return {f"p{int(q * 100)}": s[min(len(s) - 1, max(0, math.ceil(q * len(s)) - 1))] for q in qs}


def km_quantiles(done, cens, qs=(0.5, 0.9)):
    """Kaplan-Meier quantiles of tokens-per-attempt with right-censoring (aborted/unfinished attempts are censored at
    the tokens they reached). None where the survival curve never drops that far."""
    events = sorted([(t, 1) for t in done] + [(t, 0) for t in cens])
    at_risk, surv, out, i = len(events), 1.0, {}, 0
    pending = list(qs)
    while i < len(events) and pending:
        t = events[i][0]
        d = c = 0
        while i < len(events) and events[i][0] == t:
            d += events[i][1]
            c += 1 - events[i][1]
            i += 1
        if d and at_risk:
            surv *= 1 - d / at_risk
        at_risk -= d + c
        while pending and 1 - surv >= pending[0]:
            out[f"km_p{int(pending.pop(0) * 100)}"] = t
    for q in pending:
        out[f"km_p{int(q * 100)}"] = None
    return out


def token_stats(rows):
    done = [r["completion_tokens"] for r in rows if r.get("state") == "done" and r.get("completion_tokens")]
    cens = [r["completion_tokens"] for r in rows if r.get("state") != "done" and r.get("completion_tokens")]
    out = {"n": len(rows), "n_done": len(done), "n_censored": len(cens),
           "mean_done": (sum(done) / len(done)) if done else None, "max_done": max(done) if done else None,
           "sum_all": sum(done) + sum(cens), "censored_over_40k": sum(1 for t in cens if t >= 40000),
           "censored_over_80k": sum(1 for t in cens if t >= 80000)}
    out.update(quantiles(done))
    out.update(km_quantiles(done, cens))
    return out


def arm_summary(rows):
    """rows: one effort arm. Accuracy is per test output; truncation = finish_reason 'length' among finished."""
    finished = [r for r in rows if r.get("state") == "done"]
    return {"tokens": token_stats(rows),
            "truncated": sum(1 for r in finished if r.get("finish_reason") == "length"),
            "parsed": sum(1 for r in finished if r.get("answer") is not None),
            "correct": sum(1 for r in finished if r.get("correct")),
            "finished": len(finished), "requests": len(rows)}


def pair_agreement(rows):
    """For an arm sampled twice per output: agreement count and its precision (agreeing answer correct)."""
    by = {}
    for r in rows:
        if r.get("state") == "done":
            by.setdefault(r["key"], []).append(r)
    agree = agree_ok = both = 0
    for rs in by.values():
        if len(rs) < 2:
            continue
        both += 1
        a, b = rs[0].get("answer"), rs[1].get("answer")
        if a is not None and a == b:
            agree += 1
            agree_ok += bool(rs[0].get("correct"))
    return {"pairs": both, "agree": agree, "agree_correct": agree_ok,
            "precision": (agree_ok / agree) if agree else None}


In [ ]:
%%writefile progsynth.py
"""Program-synthesis channel helpers (CSR-arc-agi-2-001 #1/#3, HANDOFF 10-13; v13: sandbox hardened per API review 20261013T150520Z): prompt for a `transform(grid)` program,
extract it, run it in a resource-limited subprocess on the train inputs and the test inputs, and build one round of
execution feedback. Pure stdlib; the sandbox is a separate `python -I` process per program."""
import json
import re
import subprocess
import sys
import textwrap

import arc_llm as L

RUNNER = r'''
import json, os, resource, sys
for lim, val in ((resource.RLIMIT_AS, 2 << 30), (resource.RLIMIT_CPU, 20), (resource.RLIMIT_FSIZE, 10 << 20)):
    try:
        resource.setrlimit(lim, (val, val))
    except (ValueError, OSError):  # macOS refuses RLIMIT_AS; Linux (Colab/Kaggle) applies it
        pass
real = os.fdopen(os.dup(1), "w")  # the result channel; the model's own prints go to /dev/null (no parent OOM)
null = open(os.devnull, "w")
os.dup2(null.fileno(), 1)
os.dup2(null.fileno(), 2)
sys.stdout = sys.stderr = null
src, inputs = json.loads(sys.stdin.read())
ns = {}
out = []
try:
    exec(compile(src, "<model>", "exec"), ns)
    fn = ns["transform"]
except BaseException as e:
    real.write(json.dumps({"compile_error": repr(e)[:400]})); real.flush(); os._exit(0)
for g in inputs:
    try:
        r = fn([list(row) for row in g])
        if hasattr(r, "tolist"):
            r = r.tolist()
        r = [[int(v) for v in row] for row in r]
        out.append({"ok": r})
    except BaseException as e:
        out.append({"err": repr(e)[:300]})
real.write(json.dumps({"results": out})); real.flush()
os._exit(0)
'''


def build_prompt(task):
    parts = ["You are solving an ARC-AGI puzzle by writing a program. Each example maps an input grid to an output grid "
             "with the same hidden rule. Grids are lists of rows; each cell is a colour 0-9.", ""]
    for i, p in enumerate(task["train"], 1):
        parts += [f"Example {i} input:", L.grid_text(p["input"]), f"Example {i} output:", L.grid_text(p["output"]), ""]
    for i, p in enumerate(task["test"], 1):
        parts += [f"Test input {i} (output unknown):", L.grid_text(p["input"]), ""]
    parts += ["Write a Python function `transform(grid)` that takes one input grid (list of lists of int) and returns "
              "the output grid (list of lists of int). It must reproduce every example output exactly and generalise "
              "to the test inputs. You may use the standard library and numpy. Give the complete final code in one "
              "```python code block at the end of your answer."]
    return "\n".join(parts)


_CODE = re.compile(r"```(?:python|py)?\s*\n(.*?)```", re.S)


def extract_code(text):
    blocks = [b for b in _CODE.findall(text or "") if "def transform" in b]
    return blocks[-1] if blocks else None


def run_program(code, grids, timeout=30):
    """Returns (results, error): results is a list of {"ok": grid} / {"err": msg} per input grid. The child runs in its
    own session and an empty temp cwd, single-threaded BLAS, and is killed with its whole process group on timeout."""
    import os
    import shutil
    import signal
    import tempfile
    cwd = tempfile.mkdtemp(prefix="v13prog-")
    env = dict(os.environ, OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
    p = None
    try:
        # -s, not -I: keep the notebook's PYTHONPATH (Kaggle's numpy may live there); the cwd is an empty temp dir, so
        # nothing can be planted next to the runner
        p = subprocess.Popen([sys.executable, "-s", "-c", RUNNER], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                             stderr=subprocess.DEVNULL, text=True, cwd=cwd, env=env, start_new_session=True)
        stdout, _ = p.communicate(json.dumps([code, grids]), timeout=timeout)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(p.pid, signal.SIGKILL)
        except Exception:  # noqa: BLE001
            pass
        p.communicate()
        return None, "timeout"
    finally:
        if p is not None and p.poll() is None:
            try:
                os.killpg(p.pid, signal.SIGKILL)
            except Exception:  # noqa: BLE001
                pass
        shutil.rmtree(cwd, ignore_errors=True)
    try:
        out = json.loads(stdout.strip().splitlines()[-1])
    except Exception:  # noqa: BLE001
        return None, "no output (killed or crashed)"
    if "compile_error" in out:
        return None, out["compile_error"]
    return out["results"], None


def check(code, task):
    """Run on train + test inputs of the (augmented) task. verified = every train output reproduced exactly and every
    test output is a valid grid."""
    grids = [p["input"] for p in task["train"]] + [p["input"] for p in task["test"]]
    res, err = run_program(code, grids)
    n = len(task["train"])
    if res is None:
        return {"verified": False, "error": err, "train_pass": 0, "train_n": n, "tests": None}
    passes = [r.get("ok") == p["output"] for r, p in zip(res[:n], task["train"])]
    tests = [r.get("ok") if L.valid_grid(r.get("ok")) else None for r in res[n:]]
    return {"verified": all(passes) and all(t is not None for t in tests), "error": None, "train_pass": sum(passes),
            "train_n": n, "tests": tests, "train_results": res[:n]}


def feedback(task, chk):
    """One round of execution feedback for a failed program."""
    if chk.get("error"):
        return textwrap.shorten(f"Your code failed to run: {chk['error']}", 600) + \
            "\nFix it. Give the complete corrected code in one ```python block at the end."
    lines = [f"Your program reproduces {chk['train_pass']} of {chk['train_n']} example outputs. Mismatches:"]
    for i, (r, p) in enumerate(zip(chk["train_results"], task["train"]), 1):
        if r.get("ok") == p["output"]:
            continue
        if "err" in r:
            lines.append(f"Example {i}: error {r['err'][:200]}")
            continue
        got = r.get("ok")
        if not L.valid_grid(got) or len(got) * len(got[0]) > 400 or len(p["output"]) * len(p["output"][0]) > 400:
            shape = f"{len(got)}x{len(got[0]) if got and isinstance(got[0], list) else '?'}" if isinstance(got, list) else "?"
            lines.append(f"Example {i}: expected shape {len(p['output'])}x{len(p['output'][0])}, got {shape}")
        else:
            lines.append(f"Example {i}: expected\n{L.grid_text(p['output'])}\ngot\n{L.grid_text(got)}")
    lines.append("Find the rule that fits every example, then give the complete corrected code in one ```python block "
                 "at the end.")
    return "\n".join(lines)


In [ ]:
%%writefile v13_merge.py
"""v13 fill rule (pre-registered, HANDOFF 2026-10-13): per test output, the answer of the verified programs (majority,
ties -> first verified) becomes attempt_1 and NVARC pass A's top-1 attempt_2; when the program answer already is one of
NVARC's two attempts the entry is left unchanged; when NVARC has no real grid, attempt_2 is the next distinct program
answer. Pure functions + atomic write; no third-party imports."""
import json
import os

PLACEHOLDER = [[0]]


def valid_grid(g):
    return (isinstance(g, list) and 0 < len(g) <= 30 and all(isinstance(r, list) for r in g)
            and 0 < len(g[0]) <= 30 and all(len(r) == len(g[0]) for r in g)
            and all(type(v) is int and 0 <= v <= 9 for r in g for v in r))


def ranked_program_answers(answer_lists, i):
    """Distinct program answers for test output i, most-supported first, ties by first appearance."""
    order, count = [], {}
    for answers in answer_lists:
        try:
            g = answers[i]
        except (IndexError, TypeError):
            continue
        if not valid_grid(g):
            continue
        key = json.dumps(g)
        if key not in count:
            order.append(key)
            count[key] = 0
        count[key] += 1
    order.sort(key=lambda k: -count[k])  # stable: ties keep first appearance
    return [json.loads(k) for k in order]


def fill_entry(base_entry, programs):
    """base_entry: {'attempt_1', 'attempt_2'} from NVARC (may be placeholders); programs: ranked distinct grids."""
    if not programs:
        return dict(base_entry)
    nv = [base_entry.get(a) for a in ("attempt_1", "attempt_2")]
    nv = [g for g in nv if valid_grid(g) and g != PLACEHOLDER]
    best = programs[0]
    if best in nv:
        return dict(base_entry)
    second = nv[0] if nv else next((g for g in programs[1:] if g != best), None)
    if second is None:
        second = base_entry.get("attempt_1") if valid_grid(base_entry.get("attempt_1")) else PLACEHOLDER
    return {"attempt_1": best, "attempt_2": second}


def merge(test, base, results):
    """test: challenges dict; base: NVARC submission (same keys); results: task -> list of verified answer lists
    (one grid per test output, original frame). Returns (merged, n_changed_outputs)."""
    merged, changed = {}, 0
    for key, task in test.items():
        entries = []
        for i in range(len(task["test"])):
            try:
                b = dict(base[key][i])
            except Exception:  # noqa: BLE001  (missing/odd base entry: placeholders)
                b = {"attempt_1": PLACEHOLDER, "attempt_2": PLACEHOLDER}
            for a in ("attempt_1", "attempt_2"):
                if not valid_grid(b.get(a)):
                    b[a] = PLACEHOLDER
            e = fill_entry(b, ranked_program_answers(results.get(key, []), i))
            changed += e != b
            entries.append(e)
        merged[key] = entries
    return merged, changed


def validate(test, submission):
    if set(submission) != set(test):
        raise ValueError("submission keys differ from test keys")
    for k, task in test.items():
        entries = submission[k]
        if not isinstance(entries, list) or len(entries) != len(task["test"]):
            raise ValueError(f"{k}: wrong number of test entries")
        for e in entries:
            if not isinstance(e, dict) or set(e) != {"attempt_1", "attempt_2"}:
                raise ValueError(f"{k}: attempts missing")
            if not all(valid_grid(g) for g in e.values()):
                raise ValueError(f"{k}: invalid grid")
    return True


def atomic_json(path, value):
    tmp = str(path) + ".tmp"
    with open(tmp, "w") as f:
        json.dump(value, f)
    os.replace(tmp, path)


def load_results(path):
    """task -> list of verified answer lists, from the driver's programs jsonl (verified rows only)."""
    res = {}
    try:
        with open(path) as f:
            for line in f:
                try:
                    r = json.loads(line)
                except ValueError:
                    continue
                if r.get("verified") and isinstance(r.get("answers"), list):
                    res.setdefault(r["task"], []).append(r["answers"])
    except OSError:
        pass
    return res


def finalize(work_dir="/kaggle/working", test_path=None):
    """Last notebook cell (rerun): rebuild submission.json from the saved NVARC base + verified programs; on any
    problem keep a valid submission (current file if valid, else the base). Never raises."""
    sub = os.path.join(work_dir, "submission.json")
    base_path = os.path.join(work_dir, "v13-base-submission.json")
    record = {"finalize": None}
    try:
        with open(test_path) as f:
            test = json.load(f)
        if not os.path.exists(base_path):
            record["finalize"] = "no base (LLM phase did not start); submission.json left as assembled"
            return record
        with open(base_path) as f:
            base = json.load(f)
        merged, changed = merge(test, base, load_results(os.path.join(work_dir, "v13-programs.jsonl")))
        validate(test, merged)
        atomic_json(sub, merged)
        record.update(finalize="merged", changed_outputs=changed)
    except Exception as e:  # noqa: BLE001
        record["finalize"] = f"error {type(e).__name__}: {e}"[:300]
        try:
            with open(test_path) as f:
                test = json.load(f)
            with open(sub) as f:
                validate(test, json.load(f))
            record["kept"] = "current submission.json (valid)"
        except Exception:  # noqa: BLE001
            try:
                with open(base_path) as f:
                    base = json.load(f)
                validate(test, base)
                atomic_json(sub, base)
                record["kept"] = "base"
            except Exception as e2:  # noqa: BLE001
                record["kept"] = f"nothing valid: {type(e2).__name__}"
    finally:
        print(json.dumps({"msg": "v13_finalize", **record}), flush=True)
    return record


In [ ]:
%%writefile v13_llm.py
"""v13 LLM phase (runs after NVARC pass A has been assembled into submission.json): serve Qwen3.8-27B-FP8 with vLLM
0.19.0 from a Python 3.12 venv (python-build-standalone + the cp312 wheelhouse; the NVARC Python 3.11 stack is never
touched) and run the program-synthesis channel: write `transform(grid)`, verify it on every train pair of an augmented
copy of the task, and when it passes put its test answer into attempt_1 (v13_merge fill rule). Every verified program
rewrites the submission atomically, so a hard stop at any moment leaves a valid file.

Any failure before the first job (time, inputs missing, GPUs busy, install, server) skips the phase and keeps NVARC's
submission. Rerun: tasks from the competition test file only, unsure-first by NVARC pass-A confidence. Commit: the
4 NVARC smoke tasks + the 30 public-eval keys30 tasks, scored against the evaluation solutions (dev measurement).
  python v13_llm.py --end-time T   (T = epoch seconds by which this phase must have returned)"""
import argparse
import glob
import hashlib
import json
import os
import re
import signal
import socket
import subprocess
import sys
import threading
import time
import zlib
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
import arc_llm as L  # noqa: E402
import progsynth as P  # noqa: E402
import v13_merge as M  # noqa: E402

T0 = time.time()
A = None
S = {"version": "v13-llm-v5", "t0": T0, "phases": {}, "skipped": None}
LOCK = threading.Lock()
RESULTS = {}           # task -> list of verified answer lists (original frame)
STOP = threading.Event()
ACTIVE = {}          # open streaming responses; closed at the abort time (a preempted stream sends nothing)
SERVER = [None]        # the server process, for the last-resort kill after the abort time
PORT = int(os.environ.get("V13_PORT", "8000"))  # test seam; production 8000
MAX_LEN = 131072
PY_TGZ = "cpython-3.12.14+20260924-x86_64-unknown-linux-gnu-install_only.tar.gz"
PY_SHA = "5eae8cf79dd47fc2496a4fccc892936be831ce7a84d984b2299dfb1cdb592682"
PYPI = {  # sha256 of the official PyPI wheels (pypi.org JSON API, 2026-10-12; same pins as P0/P1)
    "vllm-0.19.0-cp38-abi3-manylinux_2_31_x86_64.whl":
        "2d0e5fae45367bdbf111fcad68f4c0f8fdddd2f2fb643e52f0f2daebef7b41cf",
    "torch-2.10.0-3-cp312-cp312-manylinux_2_28_x86_64.whl":
        "98c01b8bb5e3240426dcde1446eed6f40c778091c8544767ef1168fc663a05a6",
}
CONFIDENT = 5.0        # arc_v6_config.PASS_B_CONFIDENT: kgmon top-1 score counted as sure (v9)
MAX_REPS = 6           # augmentation repetitions per task; rep r >= 1 only for tasks without a verified program


def el():
    return round(time.time() - T0, 1)


def log(msg):
    print(f"[v13 {el():8.1f}s] {msg}", flush=True)


def save():
    with LOCK:
        text = json.dumps(S, default=str)
        tmp = os.path.join(A.work, "v13-llm.json.tmp")
        with open(tmp, "w") as f:
            f.write(text)
        os.replace(tmp, os.path.join(A.work, "v13-llm.json"))


def append(name, row):
    with LOCK, open(os.path.join(A.work, name), "a") as f:
        f.write(json.dumps(row, default=str) + "\n")


def run(cmd, timeout=None, **kw):
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, **kw)
        return p.returncode, p.stdout + p.stderr
    except Exception as e:  # noqa: BLE001
        return None, repr(e)


def sha_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""):
            h.update(b)
    return h.hexdigest()


def find(pattern):
    return sorted(glob.glob(os.path.join(A.input_root, "**", pattern), recursive=True))


class Skip(Exception):
    pass


def budget(seconds):
    """Timeout for one setup step: never past the setup deadline (server must be ready by then); skip when < 30 s."""
    left = A.setup_deadline - time.time()
    if left < 30:
        raise Skip(f"setup time exhausted ({round(left)}s left)")
    return min(seconds, left)


def venv_env():
    """Environment for every Python 3.12 venv process (pip, checks, server): nothing from the NVARC 3.11 stack leaks in
    (PYTHONPATH/HOME, user site, PIP_*), and the CUDA toolkit dirs do not shadow the torch wheel's bundled CUDA libs."""
    drop = ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV", "PYTHONSTARTUP", "PYTHONUSERBASE", "CONDA_PREFIX")
    env = {k: v for k, v in os.environ.items() if k not in drop and not k.startswith("PIP_")}
    env["PYTHONNOUSERSITE"] = "1"
    parts = [x for x in env.get("LD_LIBRARY_PATH", "").split(":") if x and not x.startswith("/usr/local/cuda")]
    libcuda = "/usr/local/nvidia/lib64/libcuda.so"
    if os.path.exists(libcuda):  # same libcuda pin as P0/P1 (public Apache-2.0 Qwen3.8 notebook)
        parts = ["/usr/local/nvidia/lib64"] + [x for x in parts if x != "/usr/local/nvidia/lib64"]
        env["LD_PRELOAD"] = libcuda
    env["LD_LIBRARY_PATH"] = ":".join(parts)
    return env


# --- 1. inputs, GPUs, install ------------------------------------------------------------------------------------------

def phase_inputs():
    ph = S["phases"]["inputs"] = {}
    wheels = find("vllm-0.19.0-*.whl")
    tgz = find(PY_TGZ)
    model = [os.path.dirname(p) for p in find("layers-0.safetensors")]
    ph.update(wheels=wheels[:1], py_tgz=tgz[:1], model=model[:2])
    if not (wheels and tgz and len(model) == 1):
        raise Skip(f"inputs missing: wheels={len(wheels)} py={len(tgz)} model={len(model)}")
    return Path(wheels[0]).parent, tgz[0], Path(model[0])


def gpu_used():
    rc, out = run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], timeout=60)
    vals = [int(x) for x in out.split() if x.strip().isdigit()] if rc == 0 else []
    return vals


def phase_gpus():
    """NVARC workers must be gone: all GPUs near-empty within 5 min, else kill orphan compute processes, else skip."""
    ph = S["phases"]["gpus"] = {}
    for attempt in range(2):
        end = min(time.time() + (300 if attempt == 0 else 120), A.setup_deadline)
        while time.time() < end:
            used = gpu_used()
            if A.fake_server or (used and all(u < 1500 for u in used)):
                ph.update(free_after=el(), used=used, attempt=attempt)
                return
            time.sleep(10)
        rc, out = run(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"], timeout=60)
        pids = [int(x) for x in out.split() if x.strip().isdigit()] if rc == 0 else []
        mine = ancestors()
        killed = []
        for pid in pids:  # only NVARC leftovers: container PIDs whose command line is a starter/multiprocessing child
            try:
                cmd = Path(f"/proc/{pid}/cmdline").read_bytes().replace(b"\0", b" ").decode(errors="ignore")
            except OSError:
                continue
            if pid not in mine and ("multiprocessing" in cmd or "starter.py" in cmd):
                try:
                    os.kill(pid, signal.SIGKILL)
                    killed.append(pid)
                except Exception:  # noqa: BLE001
                    pass
        rc2, _ = run(["pkill", "-9", "-f", "multiprocessing.spawn"], timeout=30)  # orphan NVARC workers/manager
        ph.setdefault("killed", []).append({"pids": killed, "pkill_rc": rc2, "used": gpu_used()})
        save()
    raise Skip(f"GPUs not released: {gpu_used()}")


def ancestors():
    pids, pid = set(), os.getpid()
    while pid > 1 and pid not in pids:
        pids.add(pid)
        try:
            pid = int(Path(f"/proc/{pid}/stat").read_text().rsplit(")", 1)[1].split()[1])
        except Exception:  # noqa: BLE001
            break
    return pids | {os.getppid()}


def verify_wheels(wheels):
    ph = S["phases"]["wheels"] = {}
    sums = {}
    for line in (wheels / "SHA256SUMS").read_text().splitlines():
        parts = line.split()
        if len(parts) == 2:
            sums[parts[1].lstrip("*")] = parts[0]
    files = sorted(p.name for p in wheels.glob("*.whl"))
    bad = [n for n in files if sums.get(n) != sha_file(wheels / n)]
    pins = {n: (sums.get(n) == h and (wheels / n).exists()) for n, h in PYPI.items()}
    ph.update(n_wheels=len(files), mismatched=bad, pypi_pins=pins)
    if bad or not all(pins.values()):
        raise Skip("wheel hash check failed")


def model_hash(model, out):
    """Commit only, informational (P1 already verified this pinned mirror; train-pair verification guards answers):
    sha256 of every weight file vs the HF LFS list. Never blocks the phase."""
    from concurrent.futures import ThreadPoolExecutor
    try:
        ref = json.loads((HERE / "qwen38_fp8_sha256.json").read_text())["files"]
        with ThreadPoolExecutor(max_workers=4) as pool:
            futs = {n: pool.submit(sha_file, model / n) for n in ref if (model / n).exists()}
            bad = [n for n, f in futs.items() if f.result() != ref[n][1] or (model / n).stat().st_size != ref[n][0]]
        res = dict(files=len(ref), missing=[n for n in ref if n not in futs], mismatched=bad, done=el())
    except Exception as e:  # noqa: BLE001
        res = {"error": repr(e)[:200]}
    with LOCK:
        out.update(res)


def phase_install(wheels, tgz):
    ph = S["phases"]["install"] = {}
    t = time.time()
    if sha_file(tgz) != PY_SHA:
        raise Skip("python tarball sha256 mismatch")
    root = Path(A.tmp)
    root.mkdir(parents=True, exist_ok=True)
    rc, out = run(["tar", "xzf", tgz, "-C", str(root)], timeout=budget(900))
    if rc != 0:
        raise Skip("python extract failed: " + out[-300:])
    venv = root / "venv"
    rc, out = run([str(root / "python/bin/python3.12"), "-m", "venv", str(venv)], timeout=budget(900), env=venv_env())
    if rc != 0:
        raise Skip("venv failed: " + out[-300:])
    vpy = str(venv / "bin/python")
    verify_wheels(wheels)
    rc, out = run([vpy, "-m", "pip", "install", "--no-index", "--find-links", str(wheels), "--requirement",
                   str(wheels / "requirements.lock")], timeout=budget(2400), env=venv_env())
    ph.update(pip_rc=rc, pip_tail=out[-1500:], pip_seconds=round(time.time() - t))
    if rc != 0:
        raise Skip("pip install failed")
    rc, out = run([vpy, "-c", "import json, torch, vllm, transformers; print(json.dumps({'torch': torch.__version__, "
                   "'cuda_ok': torch.cuda.is_available(), 'gpus': torch.cuda.device_count(), "
                   "'vllm': vllm.__version__, 'transformers': transformers.__version__}))"], timeout=budget(900),
                  env=venv_env())
    ph["import"] = out.strip().splitlines()[-1] if out.strip() else out
    if rc != 0 or '"cuda_ok": true' not in ph["import"]:
        raise Skip("venv torch/vllm import or CUDA check failed")
    rc2, _ = run([vpy, "-c", "import flash_attn"], timeout=budget(300), env=venv_env())
    ph["flash_attn"] = rc2 == 0
    if rc2 != 0:  # P0/P1 fix: vLLM forwards layer_idx to an attention impl that rejects it without flash-attn
        rc3, vroot = run([vpy, "-c", "import os, vllm; print(os.path.dirname(vllm.__file__))"], timeout=budget(300),
                         env=venv_env())
        vroot = vroot.strip().splitlines()[-1] if rc3 == 0 and vroot.strip() else ""
        patched, reverted = [], []
        for f in sorted(glob.glob(vroot + "/**/attention/*.py", recursive=True)) if vroot else []:
            text = Path(f).read_text()
            if "self.impl = impl_cls(" not in text or "extra_impl_args.pop('layer_idx', None)" in text:
                continue
            new = re.sub(r"^(\s*)self\.impl = impl_cls\(",
                         r"\1extra_impl_args.pop('layer_idx', None)\n\1self.impl = impl_cls(", text, count=1, flags=re.M)
            Path(f).write_text(new)
            if run([vpy, "-m", "py_compile", f], timeout=budget(120), env=venv_env())[0] != 0 \
                    or "extra_impl_args" not in text:
                Path(f).write_text(text)
                reverted.append(f)
            else:
                patched.append(f)
        ph.update(attention_patch=patched, attention_patch_reverted=reverted)
    ph["seconds"] = round(time.time() - t)
    save()
    return vpy


# --- 2. server --------------------------------------------------------------------------------------------------------

def server_env():
    return dict(venv_env(), PYTORCH_ALLOC_CONF="expandable_segments:True",
                VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS="1", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                VLLM_NO_USAGE_STATS="1", DO_NOT_TRACK="1", VLLM_LOGGING_LEVEL="INFO")  # backend: --attention-backend


def http(method, path, body=None, timeout=30):
    import requests
    r = requests.request(method, f"http://127.0.0.1:{PORT}{path}", json=body, timeout=timeout)
    return r.status_code, r.text


def wait_ready(proc, deadline):
    while time.time() < deadline:
        if proc.poll() is not None:
            return False
        try:
            if http("GET", "/v1/models", timeout=5)[0] == 200:
                return True
        except Exception:  # noqa: BLE001
            pass
        time.sleep(5)
    return False


def kill(proc):
    if proc is None or proc.poll() is not None:
        return
    try:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        proc.wait(timeout=60)
    except Exception:  # noqa: BLE001
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        except Exception:  # noqa: BLE001
            pass


def start_server(vpy, model, mtp, eager):
    if A.fake_server:  # test seam: a stub OpenAI-compatible server
        cmd = [sys.executable, A.fake_server, "--port", str(PORT)]
    else:
        cmd = [vpy, "-m", "vllm.entrypoints.openai.api_server", "--model", str(model), "--served-model-name", "m",
               "--host", "127.0.0.1", "--port", str(PORT), "--tensor-parallel-size", "4",
               "--gpu-memory-utilization", "0.93", "--max-model-len", str(MAX_LEN),
               "--max-num-seqs", str(max(48, A.conc)), "--kv-cache-dtype", "fp8", "--attention-backend",
               "TRITON_ATTN", "--reasoning-parser", "qwen3", "--trust-remote-code",
               "--limit-mm-per-prompt", json.dumps({"image": 0, "video": 0})]
        if mtp:
            cmd += ["--speculative-config", json.dumps({"method": "mtp", "num_speculative_tokens": mtp})]
        if eager:
            cmd += ["--enforce-eager"]
    path = os.path.join(A.work, f"v13-server-mtp{mtp}{'-eager' if eager else ''}.log")
    proc = subprocess.Popen(cmd, stdout=open(path, "w"), stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                            text=True, start_new_session=True, env=server_env())
    return proc, path


def smoke():
    body = {"model": "m", "messages": [{"role": "user", "content": "Reply with the single word OK."}],
            "max_tokens": 32, "temperature": 0, "chat_template_kwargs": {"enable_thinking": False}}
    code, text = http("POST", "/v1/chat/completions", body, timeout=300)
    return code == 200, text[-300:]


def phase_server(vpy, model, deadline):
    ph = S["phases"]["server"] = {"attempts": []}
    for mtp, eager in ((2, False), (0, False), (0, True)):
        if time.time() > deadline:
            break
        t = time.time()
        proc, path = start_server(vpy, model, mtp, eager)
        ready = wait_ready(proc, min(t + 25 * 60, deadline))
        att = {"mtp": mtp, "eager": eager, "ready": ready, "startup_seconds": round(time.time() - t)}
        if ready:
            ok, resp = smoke()
            att.update(smoke_ok=ok, smoke=resp)
            text = Path(path).read_text(errors="ignore")
            kv = re.findall(r"GPU KV cache size:\s*([\d,]+)\s*tokens", text)
            att["kv_tokens"] = int(kv[-1].replace(",", "")) if kv else None
            if ok:
                ph["attempts"].append(att)
                ph["chosen"] = att
                save()
                log(f"server ready mtp={mtp} eager={eager} in {att['startup_seconds']}s kv={att['kv_tokens']}")
                return proc
        att["log_tail"] = Path(path).read_text(errors="ignore")[-2500:] if os.path.exists(path) else ""
        ph["attempts"].append(att)
        save()
        kill(proc)
        time.sleep(10)
    raise Skip("no server configuration started")


# --- 3. jobs ------------------------------------------------------------------------------------------------------------

_CODE = re.compile(r"```(?:python|py)?\s*\n(.*?)```", re.S)


def code_blocks(text):
    return [b for b in _CODE.findall(text or "") if "def transform" in b]


def prompt_tokens(messages):
    try:
        code, text = http("POST", "/tokenize", {"model": "m", "messages": messages, "chat_template_kwargs":
                                                 {"reasoning_effort": A.effort, "enable_thinking": True}}, timeout=60)
        if code == 200 and json.loads(text).get("count"):
            return json.loads(text)["count"]
    except Exception:  # noqa: BLE001
        pass
    return sum(len(m["content"]) for m in messages) + 300  # characters: an upper bound for these digit grids


def chat(messages, cap, seed):
    import requests
    body = {"model": "m", "messages": messages, "max_tokens": cap, "temperature": 1.0, "top_p": 0.95, "top_k": 20,
            "min_p": 0.0, "seed": seed, "stream": True, "stream_options": {"include_usage": True, "continuous_usage_stats": True},
            "chat_template_kwargs": {"reasoning_effort": A.effort, "enable_thinking": True}}
    out = {"state": "running", "t_start": el(), "max_tokens": cap}
    content, reasoning, usage, finish = [], [], {}, None
    resp = None
    try:
        with requests.post(f"http://127.0.0.1:{PORT}/v1/chat/completions", json=body, stream=True,
                           timeout=(30, 900)) as r:
            if r.status_code != 200:
                return {**out, "state": "http_error", "error": r.text[-300:], "content": "", "reasoning": ""}
            resp = r
            with LOCK:
                ACTIVE[id(r)] = r
            if STOP.is_set():
                r.close()
            for raw in r.iter_lines(chunk_size=None):  # as bytes arrive (no 512-byte buffer); split on \r/\n only
                raw = raw.decode("utf-8", "replace")
                if STOP.is_set() or time.time() > A.abort_at:
                    out["state"] = "aborted"
                    break
                if not raw or not raw.startswith("data: ") or raw[6:].strip() == "[DONE]":
                    continue
                ev = json.loads(raw[6:])
                usage = ev.get("usage") or usage
                for ch in ev.get("choices") or []:
                    d = ch.get("delta") or {}
                    if d.get("content"):
                        content.append(d["content"])
                    rs = d.get("reasoning_content") or d.get("reasoning")
                    if rs:
                        reasoning.append(rs)
                    finish = ch.get("finish_reason") or finish
    except Exception as e:  # noqa: BLE001
        out.update(state="aborted" if STOP.is_set() else "exception", error=repr(e)[-300:])
    finally:
        if resp is not None:
            with LOCK:
                ACTIVE.pop(id(resp), None)
    if STOP.is_set() and out["state"] == "running" and not finish:
        out["state"] = "aborted"
    out.update(t_end=el(), completion_tokens=usage.get("completion_tokens"), prompt_tokens=usage.get("prompt_tokens"),
               finish_reason=finish, content="".join(content), reasoning="".join(reasoning))
    if out["state"] == "running":
        out["state"] = "done" if finish else "incomplete"
    return out


def suspicious(code, task, tests):
    """Cheap guards against programs that pass the train pairs by memorising them (API review 20261013T150520Z #12):
    a train output (>= 9 cells, >= 2 colours) written into the source, a test answer that copies a train output
    (>= 9 cells), or an identity answer on test when no train pair is identity. Returns a reason or None."""
    digits = re.sub(r"\D", "", code)
    outs = [p["output"] for p in task["train"]]
    for o in outs:
        flat = "".join(str(v) for r in o for v in r)
        if len(flat) >= 9 and len(set(flat)) >= 2 and flat in digits:
            return "train output hard-coded"
    for q, t in zip(task["test"], tests):
        if sum(len(r) for r in t) >= 9 and any(t == o for o in outs):
            return "test answer copies a train output"
        if t == q["input"] and not any(p["input"] == p["output"] for p in task["train"]):
            return "identity on test only"
    return None


def run_job(task_id, rep):
    raw = TASKS[task_id]
    aug = L.augmentation(task_id, salt=f"{A.seed_tag}-r{rep}")  # v5: seed tag (default v13 = v2 salts)
    task = L.augment_task(raw, aug)
    msgs = [{"role": "user", "content": P.build_prompt(task)}]
    n_prompt = prompt_tokens(msgs)
    cap = min(A.cap, MAX_LEN - n_prompt - 512)
    row = {"task": task_id, "rep": rep, "prompt_tokens_est": n_prompt, "cap": cap}
    if cap < 2000:
        row.update(state="prompt_too_long", verified=False)
        append("v13-programs.jsonl", row)
        return row
    r = chat(msgs, cap, zlib.crc32(f"{A.seed_tag}:{task_id}:{rep}".encode()) & 0x7FFFFFFF)
    row.update({k: r.get(k) for k in ("state", "finish_reason", "completion_tokens", "prompt_tokens", "t_start",
                                      "t_end", "error")})
    # candidates: the final answer's code first; then (truncated, aborted or unverified) the last code drafts of the
    # reasoning — zero extra tokens, and train-pair verification guards precision
    cands = [("content", c) for c in code_blocks(r["content"])[-1:]]
    cands += [("reasoning", c) for c in code_blocks(r["reasoning"])[-2:][::-1]]
    row.update(n_content_code=len(code_blocks(r["content"])), n_reasoning_code=len(code_blocks(r["reasoning"])),
               reasoning_len=len(r["reasoning"]), content_len=len(r["content"]), verified=False, tried=[])
    if r.get("finish_reason") != "stop":  # what the model was doing at the cap / abort (input to cap & effort choices)
        row["reasoning_tail"] = r["reasoning"][-2000:]
    seen = set()
    for source, code in cands:
        if code in seen:
            continue
        seen.add(code)
        chk = P.check(code, task)
        why = suspicious(code, task, chk["tests"]) if chk["verified"] else None
        row["tried"].append({"source": source, "verified": chk["verified"], "train_pass": chk["train_pass"],
                             "train_n": chk["train_n"], "error": (chk.get("error") or "")[:200], "rejected": why})
        if chk["verified"] and why is None:
            answers = [L.invert_aug(t, aug) for t in chk["tests"]]
            row.update(verified=True, source=source, answers=answers, code_tail=code[-1500:])
            if SOLUTIONS.get(task_id):
                row["correct"] = [a == s for a, s in zip(answers, SOLUTIONS[task_id])]
            break
    append("v13-programs.jsonl", row)  # the merge reads verified rows from this file
    if row["verified"]:
        with LOCK:
            RESULTS.setdefault(task_id, []).append(row["answers"])
        write_submission()
    return row


def write_submission():
    """Atomic rewrite from the NVARC base + every verified program so far (rerun: submission.json); one lock section,
    so an older snapshot can never land after a newer one."""
    with LOCK:
        merged, changed = M.merge(TEST, BASE, {k: list(v) for k, v in RESULTS.items()})
        M.validate(TEST, merged)
        M.atomic_json(os.path.join(A.work, OUT_NAME), merged)
        S["changed_outputs"] = changed


def job_order():
    """Rerun: NVARC pass-A confidence ascending (no candidate = -inf first), sure tasks (>= CONFIDENT) last; any
    error falls back to file order. Commit: smoke tasks (same rule) then keys30 in list order."""
    keys = list(TEST)
    conf = {}
    try:
        sys.path.insert(0, str(HERE))
        import arc_v6_config as cfg
        conf = cfg.task_confidence(keys, A.outputs, TEST, time.time() + 300)
    except Exception as e:  # noqa: BLE001
        S["order_error"] = f"{type(e).__name__}: {e}"[:300]
    rank = {k: i for i, k in enumerate(keys)}
    if A.commit:
        smoke = sorted((k for k in keys if k in COVERED), key=lambda k: (conf.get(k, float("-inf")), rank[k]))
        order = smoke + [k for k in KEYS30 if k in TEST and k not in smoke]
    else:
        unsure = [k for k in keys if not conf.get(k, float("-inf")) >= CONFIDENT]
        sure = [k for k in keys if conf.get(k, float("-inf")) >= CONFIDENT]
        order = sorted(unsure, key=lambda k: (conf.get(k, float("-inf")), rank[k])) + sure
    S["order"] = {"n": len(order), "first": order[:10], "conf_known": sum(k in conf for k in order),
                  "sure": sum(conf.get(k, float("-inf")) >= CONFIDENT for k in order)}
    return order


def stream_socket(r):
    """The socket under a streaming requests.Response (http.client path; stdlib, stable across urllib3 versions)."""
    for path in (("raw", "_fp", "fp", "raw", "_sock"), ("raw", "_connection", "sock")):
        obj = r
        try:
            for name in path:
                obj = getattr(obj, name)
            if obj is not None:
                return obj
        except AttributeError:
            continue
    return None


def abort_streams():
    """At the abort time: stop the channel and shut down every open stream's socket, so reads blocked on a stream that
    sends nothing (a preempted request) return now; 90 s later the server itself is killed as a last resort."""
    STOP.set()
    with LOCK:
        active = list(ACTIVE.values())
    for r in active:
        try:
            stream_socket(r).shutdown(socket.SHUT_RDWR)
        except Exception:  # noqa: BLE001
            pass
        try:
            r.close()
        except Exception:  # noqa: BLE001
            pass
    with LOCK:
        S["aborted_streams"] = len(active)
    last = threading.Timer(90, lambda: kill(SERVER[0]))
    last.daemon = True
    last.start()


def phase_jobs(order):
    """Workers pull (task, rep) from a queue that starts as rep 0 of every task in `order`; a job that ends without a
    verified program queues the same task's next rep at the back, so a task is never in flight twice and later reps
    only go to tasks still unsolved."""
    from collections import deque
    from concurrent.futures import ThreadPoolExecutor
    queue = deque((k, 0) for k in order)
    cond = threading.Condition()
    inflight = [0]
    done = []

    def worker():
        while True:
            with cond:
                while not queue and inflight[0] and not STOP.is_set() and time.time() < A.stop_new_at:
                    cond.wait(5)
                if not queue or STOP.is_set() or time.time() >= A.stop_new_at:
                    return
                k, rep = queue.popleft()
                inflight[0] += 1
            row = {"verified": False}
            try:
                row = run_job(k, rep)
                done.append(row)
            except Exception as e:  # noqa: BLE001  (one bad job never stops the channel)
                append("v13-programs.jsonl", {"task": k, "rep": rep, "state": "job_exception",
                                              "error": repr(e)[-300:], "verified": False})
            with cond:
                inflight[0] -= 1
                if not row.get("verified") and rep + 1 < MAX_REPS and row.get("state") != "prompt_too_long":
                    queue.append((k, rep + 1))
                cond.notify_all()

    with ThreadPoolExecutor(max_workers=A.conc) as pool:
        for _ in range(A.conc):
            pool.submit(worker)
    return done


def poller():
    keep = ("vllm:generation_tokens_total", "vllm:prompt_tokens_total", "vllm:num_requests_running",
            "vllm:num_requests_waiting", "vllm:kv_cache_usage_perc", "vllm:num_preemptions_total",
            "vllm:spec_decode_num_accepted_tokens_total", "vllm:spec_decode_num_draft_tokens_total",
            "vllm:prefix_cache_hits_total", "vllm:prefix_cache_queries_total")
    while not STOP.wait(30):
        try:
            code, text = http("GET", "/metrics", timeout=10)
            vals = {}
            for line in text.splitlines():
                m = re.match(r"([a-zA-Z_:]+)(\{[^}]*\})?\s+([0-9.eE+-]+)$", line)
                if m and m.group(1) in keep:
                    vals[m.group(1)] = vals.get(m.group(1), 0.0) + float(m.group(3))
            append("v13-metrics.jsonl", {"t": el(), **vals})
        except Exception:  # noqa: BLE001
            pass


def summarize(rows):
    done = [r for r in rows if r.get("state") not in (None, "prompt_too_long")]
    ver = [r for r in rows if r.get("verified")]
    s = {"jobs": len(rows), "finished_stop": sum(r.get("finish_reason") == "stop" for r in done),
         "finished_length": sum(r.get("finish_reason") == "length" for r in done),
         "aborted": sum(r.get("state") == "aborted" for r in done),
         "verified": len(ver), "verified_from_reasoning": sum(r.get("source") == "reasoning" for r in ver),
         "verified_tasks": len({r["task"] for r in ver}),
         "tokens": sum(r.get("completion_tokens") or 0 for r in rows)}
    if SOLUTIONS:
        s["verified_correct"] = sum(all(r.get("correct") or [False]) for r in ver)
        s["verified_correct_from_reasoning"] = sum(all(r.get("correct") or [False]) for r in ver
                                                   if r.get("source") == "reasoning")
    return s


TEST, BASE, TASKS, SOLUTIONS, KEYS30, COVERED, OUT_NAME = {}, {}, {}, {}, [], set(), "submission.json"


def load_tasks():
    global TEST, BASE, TASKS, SOLUTIONS, KEYS30, COVERED, OUT_NAME
    comp = Path(A.comp)
    if A.commit:  # dev measurement on public eval: smoke tasks + keys30, scored against the eval solutions
        TEST = json.loads((comp / "arc-agi_evaluation_challenges.json").read_text())
        sol = comp / "arc-agi_evaluation_solutions.json"
        SOLUTIONS = json.loads(sol.read_text()) if sol.exists() else {}
        KEYS30 = json.loads((HERE / "keys30.json").read_text())
        base_path = Path(A.work) / "eval-smoke-submission.json"
        OUT_NAME = "eval-v13-submission.json"
    else:
        TEST = json.loads((comp / "arc-agi_test_challenges.json").read_text())
        base_path = Path(A.work) / "submission.json"
        OUT_NAME = "submission.json"
    try:
        BASE = json.loads(base_path.read_text())
        if not A.commit:
            M.validate(TEST, BASE)
    except Exception as e:  # noqa: BLE001
        if not A.commit:  # never merge onto an unreadable NVARC submission (placeholders would replace its answers)
            raise Skip(f"NVARC base unreadable/invalid: {type(e).__name__}: {e}"[:300])
        BASE = {}
    TASKS = TEST
    try:
        COVERED = {n.split(".")[0].rsplit("_", 1)[0] for n in os.listdir(A.outputs) if not n.startswith("._tmp_")}
    except OSError:
        COVERED = set()
    if not A.commit:  # the merge base for finalize(); written once, before any program answer
        M.atomic_json(os.path.join(A.work, "v13-base-submission.json"), BASE)


def main():
    global A
    ap = argparse.ArgumentParser()
    ap.add_argument("--end-time", type=float, required=True)
    ap.add_argument("--work", default="/kaggle/working")
    ap.add_argument("--outputs", default=os.environ.get("ARC_V6_OUTPUTS", "/kaggle/inference_outputs"))
    ap.add_argument("--input-root", default=os.environ.get("ARC_V6_INPUT_ROOT", "/kaggle/input"))
    ap.add_argument("--tmp", default="/tmp/v13")
    ap.add_argument("--comp-dir", default=None)  # the notebook passes cfg.competition_dir()
    ap.add_argument("--conc", type=int, default=32)
    ap.add_argument("--cap", type=int, default=None)
    ap.add_argument("--seed-tag", default="v13")  # v5: fresh program-channel draws (v13 = v2 / E3 seeds)
    ap.add_argument("--conc-fixed", type=int, default=0)  # v5: >0 replaces the KV formula (it uses vLLM's 331,200)
    ap.add_argument("--effort", default="xhigh")
    ap.add_argument("--fake-server", default=None)  # test seams: stub server and shorter margins
    ap.add_argument("--need-seconds", type=float, default=None)
    ap.add_argument("--abort-margin", type=float, default=6 * 60)
    ap.add_argument("--stop-margin", type=float, default=None)
    A = ap.parse_args()
    A.commit = not bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
    A.cap = A.cap or 64000  # v2: commit = rerun cap (v1 dev run at 20k: 0/66 natural stops; P0b round-1 p50 = 64k)
    A.abort_at = A.end_time - A.abort_margin  # in-flight streams are cut here; their reasoning drafts are still tried
    A.stop_new_at = A.abort_at - (A.stop_margin if A.stop_margin is not None else (5 if A.commit else 15) * 60)
    if not re.fullmatch(r"v13[a-z0-9]{0,8}", A.seed_tag) or not 0 <= A.conc_fixed <= 24:
        raise SystemExit(f"bad --seed-tag {A.seed_tag!r} or --conc-fixed {A.conc_fixed}")
    S.update(seed_tag=A.seed_tag, conc_fixed=A.conc_fixed)
    S.update(commit=A.commit, cap=A.cap, conc=A.conc, effort=A.effort, end_time=A.end_time,
             seconds_left=round(A.end_time - time.time()))
    proc = None
    stop_poll = None
    try:
        A.comp = A.comp_dir
        if not A.comp:
            comps = [c for c in find("arc-agi_test_challenges.json") if "arc-prize" in c] or \
                find("arc-agi_test_challenges.json")
            A.comp = os.path.dirname(comps[0]) if comps else None
        if not A.comp or not os.path.exists(os.path.join(A.comp, "arc-agi_test_challenges.json")):
            raise Skip("competition files missing")
        load_tasks()
        need = A.need_seconds if A.need_seconds is not None else (25 if A.commit else 45) * 60
        A.setup_deadline = A.stop_new_at - need / 3  # server ready by then, else skip (>= need/3 of job time left)
        if A.stop_new_at - time.time() < need:
            raise Skip(f"not enough time: {round(A.stop_new_at - time.time())}s before the stop")
        wheels, tgz, model = phase_inputs()
        phase_gpus()
        if A.commit and not A.fake_server:
            S["phases"]["model_hash"] = {}
            threading.Thread(target=model_hash, args=(model, S["phases"]["model_hash"]), daemon=True).start()
        vpy = sys.executable if A.fake_server else phase_install(wheels, tgz)
        proc = phase_server(vpy, model, A.setup_deadline)
        SERVER[0] = proc
        kv = (S["phases"]["server"].get("chosen") or {}).get("kv_tokens")
        if A.conc_fixed:  # v5: measured KV peaks 0.71 (conc 13, 64k) / 0.76 (conc 10, 96k), no preemption
            A.conc = A.conc_fixed
        elif kv:  # KV-bound concurrency (v1 dev run: KV peaked at 0.64 with 22 streams at cap 20k -> 0.35*cap average)
            A.conc = max(8, min(A.conc, int(kv / (0.35 * A.cap + 6000)) + 2))
        S["conc_effective"] = A.conc
        stop_poll = threading.Thread(target=poller, daemon=True)
        stop_poll.start()
        S["sandbox_selftest"] = P.run_program("import numpy as np\ndef transform(g):\n    return np.array(g).T.tolist()\n",
                                              [[[1, 2]]])
        order = job_order()
        S["gen_start"] = el()
        save()
        log(f"jobs start: {len(order)} tasks, conc {A.conc}, cap {A.cap}, stop_new in "
            f"{round(A.stop_new_at - time.time())}s")
        watchdog = threading.Timer(max(0, A.abort_at - time.time()), abort_streams)
        watchdog.daemon = True
        watchdog.start()
        rows = phase_jobs(order)
        S["gen_seconds"] = round(el() - S["gen_start"])
        S["summary"] = summarize(rows)
        log("summary " + json.dumps(S["summary"]))
    except Skip as e:
        S["skipped"] = str(e)
        log(f"skipped: {e}")
    except Exception as e:  # noqa: BLE001
        import traceback
        S["skipped"] = f"error {type(e).__name__}: {e}"[:500]
        S["traceback"] = traceback.format_exc()[-3000:]
        log(f"error: {e}")
    finally:
        STOP.set()
        kill(proc)
        if RESULTS:
            try:
                write_submission()
            except Exception as e:  # noqa: BLE001
                S["final_write_error"] = repr(e)[:300]
        for log_path in glob.glob(os.path.join(A.work, "v13-server-*.log")):  # keep outputs small (P1: 872 MB log)
            try:
                if os.path.getsize(log_path) > 50 << 20:
                    with open(log_path, "rb") as f:
                        f.seek(-(20 << 20), 2)
                        tail = f.read()
                    with open(log_path, "wb") as f:
                        f.write(b"[truncated to last 20 MB]\n" + tail)
            except Exception:  # noqa: BLE001
                pass
        S["elapsed"] = el()
        save()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile model_profiles.py
"""Model profiles for the v13 LLM phase.

Default profile `qwen38_27b_fp8` holds the exact constants previously hard-coded in `v13_llm.py` and `build.py`.
Template profile `qwen4_27b_template` has all fields populated with `TODO_FILL` sentinels for the swap-in fields and
`launchable=False`; `v13_llm.py` refuses to run it (`raise Skip`) until every `TODO_FILL` has been replaced.
"""
import copy
import json

DEFAULT_PROFILE = "qwen38_27b_fp8"
TODO_FILL = "TODO_FILL"

_ALIASES = {
    "kaggle_model_source": "model_source",
    "model_slug": "model_source",
    "sha256_manifest": "sha_manifest",
    "manifest": "sha_manifest",
    "hash_manifest": "sha_manifest",
    "vllm_req": "vllm_version",
    "vllm_version_req": "vllm_version",
    "tp": "tensor_parallel_size",
    "tp_size": "tensor_parallel_size",
    "num_speculative_tokens": "mtp_tokens",
    "think_tag": "think_tags",
    "smoke": "smoke_body",
    "smoke_request": "smoke_body",
    "smoke_payload": "smoke_body",
}


class ProfileDict(dict):
    """Dict with key aliases and think_start_tag / think_end_tag views."""

    def _resolve(self, key):
        return _ALIASES.get(key, key)

    def __getitem__(self, key):
        if key in ("think_start_tag", "think_end_tag"):
            tags = dict.__getitem__(self, "think_tags")
            if isinstance(tags, (tuple, list)) and len(tags) == 2:
                return tags[0] if key == "think_start_tag" else tags[1]
            if isinstance(tags, dict):
                return tags.get("start" if key == "think_start_tag" else "end")
            return tags
        return dict.__getitem__(self, self._resolve(key))

    def __setitem__(self, key, value):
        dict.__setitem__(self, self._resolve(key), value)

    def __contains__(self, key):
        if key in ("think_start_tag", "think_end_tag"):
            return dict.__contains__(self, "think_tags")
        return dict.__contains__(self, self._resolve(key))

    def get(self, key, default=None):
        try:
            return self[key]
        except KeyError:
            return default


PROFILES = {
    "qwen38_27b_fp8": ProfileDict({
        "name": "qwen38_27b_fp8",
        "launchable": True,
        "model_source": "mikedan7/qwen3-8-27b-fp8-official/PyTorch/hf-fp8/1",
        "sha_manifest": "qwen38_fp8_sha256.json",
        "weight_glob": "layers-0.safetensors",
        "vllm_version": "0.19.0",
        "vllm_wheel_glob": "vllm-0.19.0-*.whl",
        "served_model_name": "m",
        "tensor_parallel_size": 4,
        "gpu_memory_utilization": 0.93,
        "max_model_len": 131072,
        "min_num_seqs": 48,
        "kv_cache_dtype": "fp8",
        "attention_backend": "TRITON_ATTN",
        "reasoning_parser": "qwen3",
        "trust_remote_code": True,
        "quantization": None,
        "dtype": None,
        "limit_mm_per_prompt": {"image": 0, "video": 0},
        "speculative_method": "mtp",
        "mtp_tokens": 2,
        "speculative_config": {"method": "mtp", "num_speculative_tokens": 2},
        "server_attempts": ((2, False), (0, False), (0, True)),
        "chat_template_kwargs": {
            "reasoning_effort_key": "reasoning_effort",
            "enable_thinking_key": "enable_thinking",
        },
        "reasoning_effort_key": "reasoning_effort",
        "enable_thinking_key": "enable_thinking",
        "think_tags": ("<think>", "</think>"),
        "reasoning_fields": ("reasoning_content", "reasoning"),
        "sampling": {"temperature": 1.0, "top_p": 0.95, "top_k": 20, "min_p": 0.0},
        "smoke_body": {
            "model": "m",
            "messages": [{"role": "user", "content": "Reply with the single word OK."}],
            "max_tokens": 32,
            "temperature": 0,
            "chat_template_kwargs": {"enable_thinking": False},
        },
    }),
    "qwen4_27b_template": ProfileDict({
        "name": "qwen4_27b_template",
        "launchable": False,
        "model_source": TODO_FILL,
        "sha_manifest": TODO_FILL,
        "weight_glob": "layers-0.safetensors",
        "vllm_version": TODO_FILL,
        "vllm_wheel_glob": "vllm-0.19.0-*.whl",
        "served_model_name": "m",
        "tensor_parallel_size": 4,
        "gpu_memory_utilization": 0.93,
        "max_model_len": 131072,
        "min_num_seqs": 48,
        "kv_cache_dtype": "fp8",
        "attention_backend": "TRITON_ATTN",
        "reasoning_parser": TODO_FILL,
        "trust_remote_code": True,
        "quantization": TODO_FILL,
        "dtype": None,
        "limit_mm_per_prompt": {"image": 0, "video": 0},
        "speculative_method": "mtp",
        "mtp_tokens": 2,
        "speculative_config": {"method": "mtp", "num_speculative_tokens": 2},
        "server_attempts": ((2, False), (0, False), (0, True)),
        "chat_template_kwargs": {
            "reasoning_effort_key": "reasoning_effort",
            "enable_thinking_key": "enable_thinking",
        },
        "reasoning_effort_key": "reasoning_effort",
        "enable_thinking_key": "enable_thinking",
        "think_tags": TODO_FILL,
        "reasoning_fields": ("reasoning_content", "reasoning"),
        "sampling": {"temperature": 1.0, "top_p": 0.95, "top_k": 20, "min_p": 0.0},
        "smoke_body": {
            "model": "m",
            "messages": [{"role": "user", "content": "Reply with the single word OK."}],
            "max_tokens": 32,
            "temperature": 0,
            "chat_template_kwargs": {"enable_thinking": False},
        },
    }),
}


def _has_todo(val):
    if isinstance(val, str):
        return TODO_FILL in val
    if isinstance(val, dict):
        return any(_has_todo(k) or _has_todo(v) for k, v in val.items())
    if isinstance(val, (list, tuple, set)):
        return any(_has_todo(x) for x in val)
    return False


def todo_fields(profile):
    return [k for k, v in profile.items() if _has_todo(v)]


def is_launchable(profile):
    todos = todo_fields(profile)
    if todos:
        return False
    if profile.get("name") == "qwen4_27b_template":
        return True
    return bool(profile.get("launchable", True))


def vllm_cmd(profile, vpy, model, port, max_len, conc, mtp, eager):
    cmd = [
        vpy, "-m", "vllm.entrypoints.openai.api_server",
        "--model", str(model),
        "--served-model-name", str(profile["served_model_name"]),
        "--host", "127.0.0.1",
        "--port", str(port),
        "--tensor-parallel-size", str(profile["tensor_parallel_size"]),
        "--gpu-memory-utilization", str(profile["gpu_memory_utilization"]),
        "--max-model-len", str(max_len),
        "--max-num-seqs", str(max(profile["min_num_seqs"], conc)),
    ]
    if profile.get("kv_cache_dtype"):
        cmd += ["--kv-cache-dtype", str(profile["kv_cache_dtype"])]
    if profile.get("attention_backend"):
        cmd += ["--attention-backend", str(profile["attention_backend"])]
    if profile.get("reasoning_parser"):
        cmd += ["--reasoning-parser", str(profile["reasoning_parser"])]
    if profile.get("quantization"):
        cmd += ["--quantization", str(profile["quantization"])]
    if profile.get("dtype"):
        cmd += ["--dtype", str(profile["dtype"])]
    if profile.get("trust_remote_code"):
        cmd += ["--trust-remote-code"]
    if profile.get("limit_mm_per_prompt") is not None:
        cmd += ["--limit-mm-per-prompt", json.dumps(profile["limit_mm_per_prompt"])]
    if mtp:
        spec = dict(profile["speculative_config"], num_speculative_tokens=mtp)
        cmd += ["--speculative-config", json.dumps(spec)]
    if eager:
        cmd += ["--enforce-eager"]
    return cmd


def build_chat_template_kwargs(profile, effort=None, enable_thinking=True):
    ctk = profile.get("chat_template_kwargs") or {}
    effort_key = ctk.get("reasoning_effort_key") or profile.get("reasoning_effort_key")
    think_key = ctk.get("enable_thinking_key") or profile.get("enable_thinking_key")
    kw = {}
    if effort is not None and effort_key:
        kw[effort_key] = effort
    if think_key:
        kw[think_key] = enable_thinking
    return kw


def smoke_payload(profile):
    return copy.deepcopy(profile["smoke_body"])


def tokenize_payload(profile, messages, effort):
    return {
        "model": profile["served_model_name"],
        "messages": messages,
        "chat_template_kwargs": build_chat_template_kwargs(profile, effort=effort, enable_thinking=True),
    }


def chat_payload(profile, messages, cap, seed, effort):
    samp = profile["sampling"]
    return {
        "model": profile["served_model_name"],
        "messages": messages,
        "max_tokens": cap,
        "temperature": samp["temperature"],
        "top_p": samp["top_p"],
        "top_k": samp["top_k"],
        "min_p": samp["min_p"],
        "seed": seed,
        "stream": True,
        "stream_options": {"include_usage": True, "continuous_usage_stats": True},
        "chat_template_kwargs": build_chat_template_kwargs(profile, effort=effort, enable_thinking=True),
    }


def extract_delta_reasoning(delta, profile):
    for field in profile.get("reasoning_fields", ("reasoning_content", "reasoning")):
        val = delta.get(field)
        if val:
            return val
    return None


def split_thinking(content, reasoning, profile=None):
    """If the server did not split reasoning from content, split on the profile's think tags."""
    if reasoning or not content:
        return content, reasoning
    prof = profile or PROFILES[DEFAULT_PROFILE]
    start_tag = prof.get("think_start_tag")
    end_tag = prof.get("think_end_tag")
    if (isinstance(start_tag, str) and isinstance(end_tag, str)
            and start_tag and end_tag and not _has_todo(start_tag) and not _has_todo(end_tag)
            and start_tag in content):
        _, rest = content.split(start_tag, 1)
        if end_tag in rest:
            think_part, after = rest.split(end_tag, 1)
            return after.lstrip("\r\n"), think_part
        return "", rest
    return content, reasoning


In [ ]:
%%writefile keys30.json
["dbff022c", "3dc255db", "2b83f449", "5545f144", "dd6b8c4b", "f931b4a8", "53fb4810", "332f06d7", "d35bdbdc", "58f5dbd5", "faa9f03d", "dfadab01", "80a900e0", "581f7754", "446ef5d2", "7b0280bc", "a251c730", "2c181942", "4e34c42c", "7ed72f31", "b6f77b65", "38007db0", "4a21e3da", "142ca369", "cb2d8a2c", "6e4f6532", "13e47133", "e12f9a14", "3e6067c3", "e3721c99"]

In [ ]:
%%writefile qwen38_fp8_sha256.json
{"source":"https://huggingface.co/api/models/Qwen/Qwen3.8-27B-FP8/tree/017b9c7af6b5689d5dd426a76e0bc077eb5ca20a","fetched_at":"2026-10-12T08:05Z","revision":"017b9c7af6b5689d5dd426a76e0bc077eb5ca20a","kaggle_mirror":"mikedan7/qwen3-8-27b-fp8-official/PyTorch/hf-fp8/1 (same file names and sizes; README says unmodified mirror at revision 017b9c7af6b5)","files":{"layers-0.safetensors":[383865448,"07f700e293baeaf3cd4240c3df1a948c4403f16961ea7979e86c8d6a9f8fd466"],"layers-1.safetensors":[383865448,"35840b5d452c6e438d000e5c1d8d1bc793d257394404689b8f9424749eee8edc"],"layers-10.safetensors":[383865472,"ab5fbc076ccd6514aba3ab67a0ac1d6701ad395d40cccc0086cf0284ef5d68c9"],"layers-11.safetensors":[372313760,"076b4ba44e9311c8fb8fdc0045f8ce9fd0c75584ade4e95510056b7db4b7ec61"],"layers-12.safetensors":[383865472,"86d8ec0b8fc8e3ffbdfd667b97050533ddf5a4d273b4fe1190a4335542c52829"],"layers-13.safetensors":[383865472,"71dd986134f4338a7b8290ed24993ad34a0f77b76a4cc060fe6403e36dcdda5b"],"layers-14.safetensors":[383865472,"95b285915cba994917b527030874227382ea3421dc7f47d804c0ca5851f039fb"],"layers-15.safetensors":[372313760,"1e87d61e77d2b802f796ab8a049b1119648c9b70d2ba1067f456e5a04449cfc3"],"layers-16.safetensors":[383865472,"8bcdbf4e2c7a8dda3043c4453a476b30a582d58aabfaa49b2829a8b66a664acb"],"layers-17.safetensors":[383865472,"571cea44717878cc8cbbfbd544acfe2d155f5b963fbf5dc67b3fa96afe840a1b"],"layers-18.safetensors":[383865472,"d8700627ca2267fb7beb5d398d6a659a9bfeeb2af76f10e6827b1e4359d59286"],"layers-19.safetensors":[372313760,"2b24cfb752bbed959c614d7fd85412ee27ab2ac1f47deef5d09d0592781400e4"],"layers-2.safetensors":[383865448,"32e63c6455ebfa10b4c36af2878eefcb6f8ccf8e947a62575444dc412c38ac1a"],"layers-20.safetensors":[383865472,"52fd00c68d4f1df96ba701290c04237257ebb832879bb0b6a629525809d84a64"],"layers-21.safetensors":[383865472,"e3c29dd949808c13116d853ce4b0f96cb1238fc0cb6999ab76735deef6bda77a"],"layers-22.safetensors":[383865472,"63446546e3e09c2394fc51855f4e390cb47f896bb8b744f3a210c3159a3ccaf2"],"layers-23.safetensors":[372313760,"4166ed11d0329985f9813ba6a70474ea9f5aeff3d97db3d7b2e81185bd41a996"],"layers-24.safetensors":[383865472,"720bdcfe12b6627bed29feda9c96256e15c20602704669d05097668a9f03dd45"],"layers-25.safetensors":[383865472,"1b188e7e4e8ae6f5d753c597f61d86c1593495de4307c273d8dda9946ec48bb4"],"layers-26.safetensors":[383865472,"ad8c3e3d4b79dfedad674318ee4b150b2cd78b56747c7c8d88a65818109a545d"],"layers-27.safetensors":[372313760,"8d63f461430045960f27a22abf685bfc7445fffbb24764e4a7fa993ba7edac08"],"layers-28.safetensors":[383865472,"f78557cda66107ce25f8c5ecde2f83da874b3bf7c50381e062e14e7294a5b64d"],"layers-29.safetensors":[383865472,"09f264d6fdddab0fb5ecce18034a9a01944d15118147c97058fba101a60f8af1"],"layers-3.safetensors":[372313744,"302f9af90bb683a8be9e96d124b470a2eddee6612c95c39a8e93f26eb654563d"],"layers-30.safetensors":[383865472,"54afe4fc7262ee597d4d9b80d2d6fe8ae501846ff124334b575bed4c6850ed97"],"layers-31.safetensors":[372313760,"2779338203d0715ca323ac0b7fdbb0d450ce2bd1694be395a211a02ade3f34d9"],"layers-32.safetensors":[383865472,"f4af8893594458b6cdb074ef20f9accb708361fe904012465990d09e74994509"],"layers-33.safetensors":[383865472,"8f038505b9fc7d0b31ca976c728b56028ec28390a15ca0410f00df18d3abc1a4"],"layers-34.safetensors":[383865472,"0b28f43cc4d3ae50e37eb3c2ce12a6e181b1c7586a27d5a05c6a3367f08286c9"],"layers-35.safetensors":[372313760,"535b61b9eabbcda5de20e21d2d46fb52528cc9c1f8fac38920382ea2f1030837"],"layers-36.safetensors":[383865472,"2babd209cafbe12a6e375ff6e7a1029f36ee8226f2a9babf6646630f31dd3fb8"],"layers-37.safetensors":[383865472,"651cae579808983fefec954db6e2ac8c3f98bfb038ff1d50aef0505ba7394e65"],"layers-38.safetensors":[383865472,"1c60932dec650cc7679121eaa4d90b8d10a600bf58e23b0784588a339469dd0a"],"layers-39.safetensors":[372313760,"b0c7df4d51b99637ff4ea35eebf7eb134db592eb943e3ccde921729f33949058"],"layers-4.safetensors":[383865448,"b7f367125cdb4b3c3920d1d3ed1ba0d73e464c3e3a92dbd1c195658cb9200afd"],"layers-40.safetensors":[383865472,"6908f1ab1a3d7828a566099ecf393582bec988d0f82f6141848942d1494b0321"],"layers-41.safetensors":[383865472,"21ee5d9842074888bd9fdffa875d1f29f8c77f1f5ccbb8f53becb188e1749caa"],"layers-42.safetensors":[383865472,"565379d4291c06cdf9a4f66d5d21385848a1e5ab17b55fea6c1c8e544f8498f0"],"layers-43.safetensors":[372313760,"4bb8c0a4fadda1f1d66ef6f0dde5a5a94001d04af1842e42009833aff70bf416"],"layers-44.safetensors":[383865472,"14339304cd520a8673f179cb8352c36fbebe184f15309aee2a79afd2d4476fef"],"layers-45.safetensors":[383865472,"7a076aca542f937cc87dec92ab49fd782eae234642b40bc706a6108b3dff2c2a"],"layers-46.safetensors":[383865472,"d82d472cfc4792012934467b90c5e0242df4867a430dd651d13cdee72b7c6645"],"layers-47.safetensors":[372313760,"a83fc757fb11bc50ed3a0c9998b2b2f55c3b6a07598ed458bdbece4285ffaf1f"],"layers-48.safetensors":[383865472,"f6aa126d006c7d976c20b1330c118b42cf3bbe8a7d216bb14f2240d5c49bc72f"],"layers-49.safetensors":[383865472,"7ed54c375407c2baec8cfabe8336f8d3782246848298bb232ac2582113676c69"],"layers-5.safetensors":[383865448,"ebbc2c4bd2b98877caafa7073b6ed732a0284eb82269cf6d21710e245af5837a"],"layers-50.safetensors":[383865472,"73825e1f056c3de53276d3c65743e62b67bf3ca082cf7077ecba1ffef18e9bf1"],"layers-51.safetensors":[372313760,"b6abd8b62b5c4b7beca8cdccd306065be2192ef42491c587ca019d971ed86a1f"],"layers-52.safetensors":[383865472,"6251d51e6fbec6f463ca875fcbe852e07c5025145affe9af053a04fbdc37288f"],"layers-53.safetensors":[383865472,"a9f2f43b4978bba2f755c6d2eb39f8561965bc3c1b2fd1073bb363aff11a9b8c"],"layers-54.safetensors":[383865472,"f0246c9974a4b1207f8f5d05a09c54c50ea61fbfd2dc6480d2b76d3647ece292"],"layers-55.safetensors":[372313760,"649c400495616cf888b7c52e538113610a36265c531d84bff484f8149c157405"],"layers-56.safetensors":[383865472,"eb0a8139a36138639a37219c158b9415412b68d316235061a72e57da5509c188"],"layers-57.safetensors":[383865472,"d7616ad33f9256342573ff85b3f2a0e368dd120257c0c7158564587036119d95"],"layers-58.safetensors":[383865472,"a8dff0e37d0e3903cc29711b935c2865ccd38bd930e7487ae2f4bedbd18ff185"],"layers-59.safetensors":[372313760,"573ed65fd455ec9bd811e95946ea422e08ac36860fdff6968a6d108263a54175"],"layers-6.safetensors":[383865448,"2fea2aaa61d566ba8af27ee89d29d9ee9f1c7f6fbe692772d9ba61fc45109ef6"],"layers-60.safetensors":[383865472,"9a19c30b190bf552eb424af16103f610bcd944fb276b3275eb7c73849799b7de"],"layers-61.safetensors":[383865472,"ff744debd4dd0eef450ed24473df57680db404914302e7147a3b72154131f4a0"],"layers-62.safetensors":[383865472,"d9b3b9e472f78bcadf62c446084833bb549faff9e730b21219f1f80d583eff23"],"layers-63.safetensors":[372313760,"59ba4a3af5e6bc2008c2d6b5d9be8eeaf255cddc0aaf8a0d9583b496bb2e3ae0"],"layers-7.safetensors":[372313744,"f2e0137e878016a7afdb2314972e0cbbe2461a67b00eb1d691b4de8ea2665da8"],"layers-8.safetensors":[383865448,"1721b7bcd730891c224b70a7b6147e44a574a2d6c4d924760acf97728249c92c"],"layers-9.safetensors":[383865448,"0a21e07065b5bb04adc339300dea1872cc48ab8eab3b02df092f7cbc67c8d6f6"],"mtp.safetensors":[477202224,"e5e4464a3793cc261de536592830bca40e7f3af159ed038c358f5660917cf43b"],"outside.safetensors":[6007102112,"ddff1d6665a2b39f2612fce0ef955e2436724c565bfbcbc127c7ffd078b698ff"],"tokenizer.json":[12809320,"0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3"]}}


In [ ]:
import json
import arc_v6_config as cfg
global_end_time = cfg.end_time(global_start)
v13_end_time = global_end_time if cfg.is_rerun() else global_start + 7200
with open(cfg.test_path()) as f:
    _test = json.load(f)
cfg.atomic_json(cfg.WORK_DIR + '/submission.json', cfg.fallback_submission(_test))
cfg.log('fallback_written', rerun=cfg.is_rerun(), test_tasks=len(_test), model_dir=cfg.model_dir(),
        competition_dir=cfg.competition_dir(),
        work_file=cfg.work_challenges_path(), seconds_left=round(global_end_time - time.time()),
        v13_seconds_left=round(v13_end_time - time.time()))

In [ ]:
!PYTHONUNBUFFERED=1 UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 ARC_V8_TRAIN_AUG_SEED=3001 ARC_V8_EVAL_AUG_SEED=3002 ARC_V8_TRAINER_SEED=3042 ARC_V8_LORA_SEED=3042 ARC_V8_SCORE_SEED_OFFSET=3000 ARC_V8_PASS_B_MIN_SECONDS=1e12 python starter.py --end-time {global_end_time}

In [ ]:
import assemble
record = assemble.main()

In [ ]:
_comp_dir = cfg.competition_dir()
!PYTHONUNBUFFERED=1 python v13_llm.py --end-time {v13_end_time} --comp-dir {_comp_dir} --seed-tag v13s5 --conc-fixed 16

In [ ]:
import v13_merge
if cfg.is_rerun():
    v13_merge.finalize(cfg.WORK_DIR, cfg.test_path())
with open(cfg.WORK_DIR + '/submission.json') as f:
    cfg.validate_submission(_test, json.load(f))
cfg.log('v13_done', seconds_left=round(global_end_time - time.time()))